# **Notebook 7: Solution V2 — Fine-Tuned RAG Evaluation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] `./intent_lora_best/` — Created by Notebook 6
- [ ] `./chroma_db/` — Created by Notebook 4
- [ ] `df_test.csv` — Created by Notebook 2
- [ ] `outputs.json` + `v1_metrics.csv` — From Notebooks 3/4/5
- [ ] GPU runtime enabled

**Files this notebook will CREATE:**
- [ ] `Comparative_Results_Full.csv` + `Comparative_Results_Summary.csv` _(Final deliverables)_

---

In [1]:
# ============================================================
# COLAB PROJECT SETUP
# ============================================================

from google.colab import drive
from pathlib import Path
import os

# Mount Google Drive
drive.mount("/content/drive")

# Permanent project folder in Google Drive
DRIVE_PROJECT_DIR = Path(
    "/content/drive/MyDrive/Hybrid_RAG_Customer_Support"
)

# Temporary workspace for the current Colab runtime
LOCAL_PROJECT_DIR = Path(
    "/content/Hybrid_RAG_Customer_Support"
)

LOCAL_PROJECT_DIR.mkdir(parents=True, exist_ok=True)

# Work from the temporary Colab directory
os.chdir(LOCAL_PROJECT_DIR)

print("Google Drive project:", DRIVE_PROJECT_DIR)
print("Local Colab workspace:", LOCAL_PROJECT_DIR)
print("Current working directory:", Path.cwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive project: /content/drive/MyDrive/Hybrid_RAG_Customer_Support
Local Colab workspace: /content/Hybrid_RAG_Customer_Support
Current working directory: /content/Hybrid_RAG_Customer_Support


In [2]:
# ============================================================
# VERIFY GPU RUNTIME
# ============================================================

import torch

print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU runtime is not enabled. In Colab, go to "
        "Runtime → Change runtime type → T4 GPU."
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gb = (
    torch.cuda.get_device_properties(0).total_memory
    / 1024**3
)

print("GPU detected    :", gpu_name)
print(f"GPU memory      : {gpu_memory_gb:.2f} GB")
print("CUDA version    :", torch.version.cuda)
print("\nGPU runtime is enabled successfully.")

PyTorch version : 2.11.0+cu128
CUDA available  : True
GPU detected    : Tesla T4
GPU memory      : 14.56 GB
CUDA version    : 12.8

GPU runtime is enabled successfully.


### **Task 4.3: Integrate Fine-Tuned Model with Retrieval**

#### **4.3.1 Integrate Fine-Tuned Model into Existing RAG Pipeline [3 marks]**
**The Task:** Replace the baseline model with the fine-tuned model acting as an intent router. Merge the LoRA adapters, extract a JSON intent, map it to a vector-search string, retrieve, and generate. Validate the integrated system.

**Hints & Tips:**
* `PeftModel.from_pretrained(base_model, "./intent_lora_best").merge_and_unload()` fuses the adapters for fast inference.
* Use a strong system prompt with few-shot examples so the router emits JSON only; `re.search(r'\{.*?\}', raw)` is a safety net for stray preamble.
* Map the intent (e.g. `track_order`) to an SOP header search string (e.g. `# Track Order`). Fall back to the raw query if JSON parsing fails.
* Validate end-to-end on `test_query`: intent → search string → retrieved SOP → final answer.

**Parameter Tuning:**
* `max_new_tokens=30` for the router (JSON is short — more tokens invite trailing explanation text).
* 4 few-shot examples is the sweet spot.

**Learner Inference:** Querying with the structured intent keyword instead of the noisy prompt retrieves the exact policy clause — the core of Hybrid RAG.

In [3]:
# YOUR CODE HERE
constraints = """
requests==2.32.4
fsspec==2025.3.0
numpy>=1.26,<2.1
websockets>=15.0.1,<16
protobuf>=5.26.1,<6
rich>=10.11,<14
tokenizers==0.21.4
opentelemetry-api==1.42.1
opentelemetry-sdk==1.42.1
opentelemetry-semantic-conventions==0.63b1
opentelemetry-exporter-otlp-proto-grpc==1.42.1
opentelemetry-exporter-otlp-proto-common==1.42.1
opentelemetry-proto==1.42.1
"""

with open("/tmp/colab_constraints.txt", "w") as file:
    file.write(constraints)

%pip install -q \
    --constraint /tmp/colab_constraints.txt \
    "chromadb==1.0.20" \
    "sentence-transformers==3.4.1" \
    "transformers==4.53.2" \
    "tokenizers==0.21.4" \
    "peft==0.16.0" \
    "bitsandbytes==0.46.1" \
    "accelerate==1.8.1" \
    "jedi>=0.16"

In [4]:
import json
import random
import re
from pathlib import Path

import chromadb
import numpy as np
import pandas as pd
import torch

from peft import (
    PeftConfig,
    PeftModel
)
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)

# Reproducibility
RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
torch.cuda.manual_seed_all(RANDOM_STATE)

if not torch.cuda.is_available():
    raise RuntimeError(
        "A T4 or better GPU runtime is required."
    )

# Define required artefact paths
BEST_ADAPTER_PATH = (
    DRIVE_PROJECT_DIR
    / "artifacts"
    / "notebook_6"
    / "intent_lora_best"
)

CHROMA_DB_PATH = (
    DRIVE_PROJECT_DIR
    / "artifacts"
    / "notebook_4"
    / "chroma_db"
)

DF_TEST_PATH = (
    DRIVE_PROJECT_DIR
    / "artifacts"
    / "notebook_2"
    / "df_test.csv"
)

OUTPUTS_PATH = (
    DRIVE_PROJECT_DIR
    / "artifacts"
    / "notebook_3"
    / "outputs.json"
)

for required_path in [
    BEST_ADAPTER_PATH,
    CHROMA_DB_PATH,
    DF_TEST_PATH,
    OUTPUTS_PATH
]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required artefact was not found:\n{required_path}"
        )

# Load test labels and the saved baseline query
df_test = pd.read_csv(DF_TEST_PATH)

required_test_columns = {
    "instruction",
    "intent",
    "category"
}

missing_columns = (
    required_test_columns
    - set(df_test.columns)
)

if missing_columns:
    raise ValueError(
        f"df_test.csv is missing: {sorted(missing_columns)}"
    )

with open(
    OUTPUTS_PATH,
    "r",
    encoding="utf-8"
) as file:
    saved_outputs = json.load(file)

test_query = saved_outputs["test_query"]
ground_truth = saved_outputs["ground_truth"]

VALID_INTENTS = set(
    df_test["intent"]
    .astype(str)
    .str.strip()
    .str.lower()
)

VALID_CATEGORIES = set(
    df_test["category"]
    .astype(str)
    .str.strip()
    .str.upper()
)

intent_to_category = (
    df_test
    .assign(
        intent=lambda frame: (
            frame["intent"]
            .astype(str)
            .str.strip()
            .str.lower()
        ),
        category=lambda frame: (
            frame["category"]
            .astype(str)
            .str.strip()
            .str.upper()
        )
    )
    .groupby("intent")["category"]
    .first()
    .to_dict()
)

assert len(VALID_INTENTS) == 27

# Recover the base model ID from the adapter
peft_config = PeftConfig.from_pretrained(
    str(BEST_ADAPTER_PATH)
)

MODEL_ID = peft_config.base_model_name_or_path

print("Recovered adapter configuration")
print("-" * 60)
print(f"Base model       : {MODEL_ID}")
print(f"Adapter path     : {BEST_ADAPTER_PATH}")
print(f"Test intents     : {len(VALID_INTENTS)}")
print(f"Test categories  : {len(VALID_CATEGORIES)}")

Recovered adapter configuration
------------------------------------------------------------
Base model       : Qwen/Qwen2.5-1.5B-Instruct
Adapter path     : /content/drive/MyDrive/Hybrid_RAG_Customer_Support/artifacts/notebook_6/intent_lora_best
Test intents     : 27
Test categories  : 11


In [5]:
# Load and merge the best LoRA router
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

router_base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

router_peft_model = PeftModel.from_pretrained(
    router_base_model,
    str(BEST_ADAPTER_PATH),
    is_trainable=False
)

# Fuse the best LoRA weights into the router for inference
router_model = router_peft_model.merge_and_unload()

router_model.config.use_cache = True
router_model.eval()

ROUTER_DEVICE = next(
    router_model.parameters()
).device

# Load an unchanged 4-bit base model for final synthesis
generator_quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

generator_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=generator_quantization,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

generator_model.eval()

GENERATOR_DEVICE = next(
    generator_model.parameters()
).device

# Reload the embedding model and persistent Chroma index
retrieval_metadata = saved_outputs[
    "naive_rag_retrieval"
]

EMBEDDING_MODEL_ID = retrieval_metadata.get(
    "embedding_model_id",
    "sentence-transformers/all-MiniLM-L6-v2"
)

COLLECTION_NAME = retrieval_metadata.get(
    "collection_name",
    "corporate_policy_chunks"
)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_ID,
    device="cpu"
)

sample_embedding = embedding_model.encode(
    ["embedding validation"],
    normalize_embeddings=True
)

assert sample_embedding.shape == (1, 384)

chroma_client = chromadb.PersistentClient(
    path=str(CHROMA_DB_PATH)
)

chroma_collection = chroma_client.get_collection(
    name=COLLECTION_NAME
)

assert chroma_collection.count() > 0

# Setup confirmation
print("\nSolution V2 components loaded")
print("=" * 60)
print(f"GPU                  : {torch.cuda.get_device_name(0)}")
print(f"Router device        : {ROUTER_DEVICE}")
print(f"Generator device     : {GENERATOR_DEVICE}")
print(f"Embedding model      : {EMBEDDING_MODEL_ID}")
print(f"Chroma collection    : {COLLECTION_NAME}")
print(f"Indexed chunks       : {chroma_collection.count()}")
print("Best LoRA adapter merged successfully.")

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Solution V2 components loaded
GPU                  : Tesla T4
Router device        : cuda:0
Generator device     : cuda:0
Embedding model      : sentence-transformers/all-MiniLM-L6-v2
Chroma collection    : corporate_policy_chunks
Indexed chunks       : 65
Best LoRA adapter merged successfully.


In [6]:
# YOUR CODE HERE
ROUTER_MAX_NEW_TOKENS = 30
ANSWER_MAX_NEW_TOKENS = 120
V2_TOP_K = 1

# Intent-to-policy search mapping
INTENT_TO_SEARCH_STRING = {
    "cancel_order":
        "order cancellation and product return policy",

    "change_order":
        "order modification and order tracking policy",

    "change_shipping_address":
        "shipping address change and order tracking policy",

    "check_cancellation_fee":
        "cancellation fees and subscription cancellation policy",

    "check_invoice":
        "invoice checking and billing dispute policy",

    "check_payment_methods":
        "accepted payment methods policy",

    "check_refund_policy":
        "refund eligibility and refund policy",

    "complaint":
        "customer complaint escalation matrix",

    "contact_customer_service":
        "customer support contact information and working hours",

    "contact_human_agent":
        "human agent escalation and escalation matrix",

    "create_account":
        "account creation and account access policy",

    "delete_account":
        "account deletion and data privacy policy",

    "delivery_options":
        "shipping options and delivery policy",

    "delivery_period":
        "shipping delays and delivery timeframe policy",

    "edit_account":
        "account profile update and account access policy",

    "get_invoice":
        "invoice retrieval and billing policy",

    "get_refund":
        "refund request and refund policy",

    "newsletter_subscription":
        "newsletter and subscription management policy",

    "payment_issue":
        "billing disputes and payment issue policy",

    "place_order":
        "order placement and product ordering policy",

    "recover_password":
        "password reset and account recovery policy",

    "registration_problems":
        "account registration and account recovery policy",

    "review":
        "customer feedback and complaint escalation policy",

    "set_up_shipping_address":
        "shipping address setup and order policy",

    "switch_account":
        "switch account and account access policy",

    "track_order":
        "order tracking and shipment status policy",

    "track_refund":
        "refund tracking and refund policy"
}

missing_intent_mappings = (
    VALID_INTENTS
    - set(INTENT_TO_SEARCH_STRING)
)

assert not missing_intent_mappings, (
    f"Missing search mappings: {sorted(missing_intent_mappings)}"
)

# Shared deterministic generation helper
def generate_text(
    model,
    device,
    messages,
    max_new_tokens
):
    """Generate one deterministic response."""

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        add_special_tokens=False
    ).to(device)

    with torch.inference_mode():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    new_token_ids = generated_ids[
        0,
        model_inputs["input_ids"].shape[1]:
    ]

    return tokenizer.decode(
        new_token_ids,
        skip_special_tokens=True
    ).strip()

# Build four few-shot router examples
few_shot_data = [
    (
        "My card was charged twice for the same order.",
        "payment_issue"
    ),
    (
        "Where is my order? It has not arrived yet.",
        "track_order"
    ),
    (
        "I forgot my password and cannot log in.",
        "recover_password"
    ),
    (
        "What is your policy for receiving a refund?",
        "check_refund_policy"
    )
]

ROUTER_SYSTEM_PROMPT = (
    "You are a customer-support intent classification assistant. "
    "Return only a valid JSON object containing exactly two keys: "
    "'intent' and 'category'. Do not include explanations, Markdown, "
    "code fences, or any text outside the JSON object."
)


def build_router_messages(query):
    """Build the router prompt with four labelled examples."""

    messages = [
        {
            "role": "system",
            "content": ROUTER_SYSTEM_PROMPT
        }
    ]

    for example_query, example_intent in few_shot_data:
        example_category = intent_to_category[
            example_intent
        ]

        messages.extend([
            {
                "role": "user",
                "content": example_query
            },
            {
                "role": "assistant",
                "content": json.dumps(
                    {
                        "intent": example_intent,
                        "category": example_category
                    },
                    separators=(",", ":")
                )
            }
        ])

    messages.append({
        "role": "user",
        "content": str(query).strip()
    })

    return messages

# 4. Parse and validate router JSON
def extract_router_json(raw_output):
    """
    Extract JSON using direct parsing first and regex as a
    safety net for accidental preamble or trailing text.
    """

    raw_output = str(raw_output).strip()

    parsing_error = None
    parsed = None

    try:
        parsed = json.loads(raw_output)

    except json.JSONDecodeError as direct_error:
        parsing_error = str(direct_error)

        json_match = re.search(
            r"\{.*?\}",
            raw_output,
            flags=re.DOTALL
        )

        if json_match:
            try:
                parsed = json.loads(
                    json_match.group(0)
                )
                parsing_error = None

            except json.JSONDecodeError as regex_error:
                parsing_error = str(regex_error)

    if not isinstance(parsed, dict):
        return {
            "valid": False,
            "intent": None,
            "category": None,
            "error": parsing_error or "No JSON object found"
        }

    if set(parsed.keys()) != {
        "intent",
        "category"
    }:
        return {
            "valid": False,
            "intent": None,
            "category": None,
            "error": (
                "Router JSON must contain exactly "
                "'intent' and 'category'."
            )
        }

    intent = str(
        parsed["intent"]
    ).strip().lower()

    category = str(
        parsed["category"]
    ).strip().upper()

    if intent not in VALID_INTENTS:
        return {
            "valid": False,
            "intent": intent,
            "category": category,
            "error": f"Unknown intent: {intent}"
        }

    if category not in VALID_CATEGORIES:
        return {
            "valid": False,
            "intent": intent,
            "category": category,
            "error": f"Unknown category: {category}"
        }

    expected_category = intent_to_category[
        intent
    ]

    if category != expected_category:
        return {
            "valid": False,
            "intent": intent,
            "category": category,
            "error": (
                f"Intent '{intent}' belongs to "
                f"category '{expected_category}', not '{category}'."
            )
        }

    return {
        "valid": True,
        "intent": intent,
        "category": category,
        "error": None
    }


def route_customer_query(query):
    """Run the fine-tuned intent router."""

    raw_router_output = generate_text(
        model=router_model,
        device=ROUTER_DEVICE,
        messages=build_router_messages(query),
        max_new_tokens=ROUTER_MAX_NEW_TOKENS
    )

    parsed_router_output = extract_router_json(
        raw_router_output
    )

    return {
        "raw_output": raw_router_output,
        **parsed_router_output
    }

In [7]:
# Map router output to a structured vector-search string
def build_structured_search_query(
    customer_query,
    router_result
):
    """
    Use the predicted intent when valid. Fall back to the raw
    customer query if JSON parsing or validation fails.
    """

    if not router_result["valid"]:
        return {
            "search_query": str(customer_query).strip(),
            "fallback_used": True
        }

    intent = router_result["intent"]
    category = router_result["category"]

    policy_search_phrase = (
        INTENT_TO_SEARCH_STRING[intent]
    )

    structured_query = (
        f"{policy_search_phrase}. "
        f"Category: {category}. "
        f"Intent: {intent}. "
        f"Customer issue: {str(customer_query).strip()}"
    )

    return {
        "search_query": structured_query,
        "fallback_used": False
    }

# Retrieve using the structured intent search
def retrieve_v2_context(
    customer_query,
    router_result,
    k=V2_TOP_K
):
    """Retrieve SOP chunks using the router-derived search text."""

    search_result = build_structured_search_query(
        customer_query=customer_query,
        router_result=router_result
    )

    search_query = search_result[
        "search_query"
    ]

    query_embedding = embedding_model.encode(
        [search_query],
        normalize_embeddings=True
    ).astype(np.float32)

    results = chroma_collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=k,
        include=[
            "documents",
            "metadatas",
            "distances"
        ]
    )

    documents = results["documents"][0]
    metadatas = results["metadatas"][0]
    distances = results["distances"][0]

    if not documents:
        raise RuntimeError(
            "Structured retrieval returned no SOP chunks."
        )

    retrieved_items = []

    for rank, (
        document,
        metadata,
        distance
    ) in enumerate(
        zip(documents, metadatas, distances),
        start=1
    ):
        retrieved_items.append({
            "rank": rank,
            "context": document,
            "source_file": metadata.get(
                "source_file",
                "unknown"
            ),
            "chunk_id": metadata.get(
                "chunk_id",
                "unknown"
            ),
            "distance": float(distance)
        })

    return {
        "search_query": search_query,
        "fallback_used": search_result["fallback_used"],
        "retrieved_items": retrieved_items
    }

# Generate the final SOP-grounded answer
def synthesise_v2_answer(
    customer_query,
    router_result,
    retrieval_result
):
    """Generate the final answer using the unchanged base model."""

    context_blocks = []

    for item in retrieval_result[
        "retrieved_items"
    ]:
        context_blocks.append(
            f"[Source: {item['source_file']} | "
            f"Chunk: {item['chunk_id']}]\n"
            f"{item['context']}"
        )

    combined_context = "\n\n".join(
        context_blocks
    )

    router_description = (
        f"Intent: {router_result['intent']}\n"
        f"Category: {router_result['category']}"
        if router_result["valid"]
        else "Intent routing failed; raw-query fallback was used."
    )

    messages = [
        {
            "role": "system",
            "content": (
                "You are a customer-support assistant. "
                "Answer strictly using only the supplied SOP context. "
                "Do not invent timelines, procedures, departments, "
                "contact details, refunds, or guarantees. If the "
                "context is insufficient, state that the available "
                "policy does not provide enough information."
            )
        },
        {
            "role": "user",
            "content": f"""
Customer query:
{str(customer_query).strip()}

Router result:
{router_description}

Retrieved SOP context:
{combined_context}

Provide a concise policy-grounded answer.
""".strip()
        }
    ]

    final_answer = generate_text(
        model=generator_model,
        device=GENERATOR_DEVICE,
        messages=messages,
        max_new_tokens=ANSWER_MAX_NEW_TOKENS
    )

    return {
        "final_answer": final_answer,
        "combined_context": combined_context
    }

# Complete integrated Solution V2 workflow
def run_solution_v2(
    query,
    k=V2_TOP_K
):
    """Run router → mapping → retrieval → final synthesis."""

    router_result = route_customer_query(
        query
    )

    retrieval_result = retrieve_v2_context(
        customer_query=query,
        router_result=router_result,
        k=k
    )

    synthesis_result = synthesise_v2_answer(
        customer_query=query,
        router_result=router_result,
        retrieval_result=retrieval_result
    )

    top_item = retrieval_result[
        "retrieved_items"
    ][0]

    return {
        "query": str(query).strip(),
        "router_raw_output": router_result["raw_output"],
        "router_valid": router_result["valid"],
        "predicted_intent": router_result["intent"],
        "predicted_category": router_result["category"],
        "router_error": router_result["error"],
        "fallback_used": retrieval_result["fallback_used"],
        "search_query": retrieval_result["search_query"],
        "retrieved_source": top_item["source_file"],
        "retrieved_chunk_id": top_item["chunk_id"],
        "retrieval_distance": top_item["distance"],
        "retrieved_context": synthesis_result["combined_context"],
        "final_answer": synthesis_result["final_answer"]
    }

In [8]:
# Validate the complete system using Notebook 3's query
v2_validation_result = run_solution_v2(
    query=test_query,
    k=V2_TOP_K
)

assert v2_validation_result["final_answer"]
assert v2_validation_result["retrieved_source"] != "unknown"
assert v2_validation_result["retrieved_context"]

if v2_validation_result["router_valid"]:
    assert (
        v2_validation_result["predicted_intent"]
        in VALID_INTENTS
    )
    assert (
        v2_validation_result["predicted_category"]
        in VALID_CATEGORIES
    )


print("Integrated Solution V2 validation")
print("=" * 70)

print(f"Customer query:\n{test_query}")

print("\nGround-truth policy statement:")
print(ground_truth)

print("\nRaw router output:")
print(v2_validation_result["router_raw_output"])

print("\nParsed router result")
print("-" * 50)
print(
    f"JSON valid         : "
    f"{v2_validation_result['router_valid']}"
)
print(
    f"Predicted intent   : "
    f"{v2_validation_result['predicted_intent']}"
)
print(
    f"Predicted category : "
    f"{v2_validation_result['predicted_category']}"
)
print(
    f"Fallback used      : "
    f"{v2_validation_result['fallback_used']}"
)
print(
    f"Router error       : "
    f"{v2_validation_result['router_error']}"
)

print("\nStructured vector-search query:")
print(v2_validation_result["search_query"])

print("\nRetrieval result")
print("-" * 50)
print(
    f"Retrieved source   : "
    f"{v2_validation_result['retrieved_source']}"
)
print(
    f"Retrieved chunk    : "
    f"{v2_validation_result['retrieved_chunk_id']}"
)
print(
    f"Retrieval distance : "
    f"{v2_validation_result['retrieval_distance']:.4f}"
)

print("\nFinal Solution V2 answer:")
print(v2_validation_result["final_answer"])

The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:447: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Integrated Solution V2 validation
Customer query:
My order was supposed to arrive already, but it is still not here. What is happening and when will I receive it?

Ground-truth policy statement:
Domestic orders are delivered within 3–7 business days.

Raw router output:
{"intent":"track_order","category":"ORDER"}

Parsed router result
--------------------------------------------------
JSON valid         : True
Predicted intent   : track_order
Predicted category : ORDER
Fallback used      : False
Router error       : None

Structured vector-search query:
order tracking and shipment status policy. Category: ORDER. Intent: track_order. Customer issue: My order was supposed to arrive already, but it is still not here. What is happening and when will I receive it?

Retrieval result
--------------------------------------------------
Retrieved source   : order_tracking.md
Retrieved chunk    : order_tracking_chunk_001
Retrieval distance : 0.3941

Final Solution V2 answer:
Based on your order's

### **Task 4.4: Evaluate Solution V2**

#### **4.4.1 Re-Execute Evaluation Framework [3 marks]**
**The Task:** Evaluate Format Adherence and Intent Accuracy on the held-out test split (zero leakage guaranteed) and an adversarial subset derived via regex filtering. Evaluate the final synthesis using ROUGE/BLEU.

**Hints & Tips:**
* Reuse `df_test` from Notebook 2 — it's the leakage-free test split.
* Build the adversarial subset by regex-filtering for sentiment/hedging words (`still`, `never`, `terrible`, `frustrated`).
* Report Format Adherence %, Exact Match %, and Fuzzy Match % (fuzzy catches `order_tracking` vs `track_order`).

**Learner Inference:** Using the held-out test split guarantees zero leakage and trustworthy scores.

In [9]:
# YOUR CODE HERE
%pip install -q rouge-score nltk rapidfuzz

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 32.8 MB/s eta 0:00:00


In [10]:
import json
import re
import time

import numpy as np
import pandas as pd

from nltk.stem import PorterStemmer
from nltk.translate.bleu_score import (
    SmoothingFunction,
    sentence_bleu
)
from rapidfuzz import fuzz
from rouge_score import rouge_scorer
from tqdm.auto import tqdm
from IPython.display import display

# Evaluation configuration and output paths
V2_EVAL_SAMPLE_SIZE = 50
REFERENCE_TOP_K = 3
FUZZY_MATCH_THRESHOLD = 80
RANDOM_STATE = 42

NOTEBOOK_7_ARTIFACT_DIR = (
    DRIVE_PROJECT_DIR
    / "artifacts"
    / "notebook_7"
)

ROUTER_CHECKPOINT_PATH = (
    NOTEBOOK_7_ARTIFACT_DIR
    / "router_evaluation_checkpoint.csv"
)

SYNTHESIS_CHECKPOINT_PATH = (
    NOTEBOOK_7_ARTIFACT_DIR
    / "v2_synthesis_checkpoint.csv"
)

NOTEBOOK_7_ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Strict router-format validation
def validate_strict_router_format(raw_output):
    """
    Validate that the complete router output is a JSON object
    containing exactly intent and category.

    Unlike the safety-net parser, this does not recover JSON
    from explanatory text. Therefore, it measures strict format
    adherence.
    """

    raw_output = str(raw_output).strip()

    if not raw_output:
        return {
            "strict_format_valid": False,
            "strict_format_error": "Empty router output",
            "strict_intent": None,
            "strict_category": None
        }

    try:
        parsed_output = json.loads(raw_output)

    except json.JSONDecodeError as error:
        return {
            "strict_format_valid": False,
            "strict_format_error": (
                f"JSONDecodeError: {error}"
            ),
            "strict_intent": None,
            "strict_category": None
        }

    if not isinstance(parsed_output, dict):
        return {
            "strict_format_valid": False,
            "strict_format_error": (
                "Router output is not a JSON object"
            ),
            "strict_intent": None,
            "strict_category": None
        }

    if set(parsed_output.keys()) != {
        "intent",
        "category"
    }:
        return {
            "strict_format_valid": False,
            "strict_format_error": (
                "JSON must contain exactly intent and category"
            ),
            "strict_intent": None,
            "strict_category": None
        }

    intent = str(
        parsed_output["intent"]
    ).strip()

    category = str(
        parsed_output["category"]
    ).strip()

    if intent != intent.lower():
        return {
            "strict_format_valid": False,
            "strict_format_error": (
                "Intent is not lowercase"
            ),
            "strict_intent": intent,
            "strict_category": category
        }

    if category != category.upper():
        return {
            "strict_format_valid": False,
            "strict_format_error": (
                "Category is not uppercase"
            ),
            "strict_intent": intent,
            "strict_category": category
        }

    return {
        "strict_format_valid": True,
        "strict_format_error": None,
        "strict_intent": intent,
        "strict_category": category
    }

# Fuzzy intent matching
stemmer = PorterStemmer()

INTENT_TOKEN_ALIASES = {
    "tracking": "track",
    "tracked": "track",
    "cancellation": "cancel",
    "cancelled": "cancel",
    "canceled": "cancel",
    "payments": "payment",
    "billing": "payment",
    "invoices": "invoice",
    "shipping": "ship",
    "delivery": "deliver",
    "registration": "register",
    "problems": "problem"
}


def normalise_intent_for_fuzzy_match(intent):
    """
    Normalise and stem intent tokens before fuzzy comparison.

    This permits near-equivalent labels such as order_tracking
    and track_order to receive a high match score.
    """

    tokens = re.split(
        r"[_\-\s]+",
        str(intent).strip().lower()
    )

    normalised_tokens = []

    for token in tokens:

        token = INTENT_TOKEN_ALIASES.get(
            token,
            token
        )

        normalised_tokens.append(
            stemmer.stem(token)
        )

    return " ".join(
        sorted(normalised_tokens)
    )


def calculate_fuzzy_intent_score(
    predicted_intent,
    true_intent
):
    """Return a token-aware fuzzy intent score from 0 to 100."""

    if predicted_intent is None:
        return 0.0

    predicted_normalised = (
        normalise_intent_for_fuzzy_match(
            predicted_intent
        )
    )

    true_normalised = (
        normalise_intent_for_fuzzy_match(
            true_intent
        )
    )

    return float(
        fuzz.token_set_ratio(
            predicted_normalised,
            true_normalised
        )
    )

# Evaluate the router on the complete held-out test set
router_evaluation_records = []
router_evaluation_start = time.perf_counter()

for test_row_id, row in tqdm(
    df_test.reset_index(drop=True).iterrows(),
    total=len(df_test),
    desc="Evaluating fine-tuned router"
):
    query = str(row["instruction"]).strip()

    true_intent = (
        str(row["intent"])
        .strip()
        .lower()
    )

    true_category = (
        str(row["category"])
        .strip()
        .upper()
    )

    try:
        router_result = route_customer_query(
            query
        )

        strict_result = (
            validate_strict_router_format(
                router_result["raw_output"]
            )
        )

        predicted_intent = (
            router_result["intent"]
        )

        predicted_category = (
            router_result["category"]
        )

        exact_intent_match = (
            router_result["valid"]
            and predicted_intent == true_intent
        )

        exact_category_match = (
            router_result["valid"]
            and predicted_category == true_category
        )

        fuzzy_score = (
            calculate_fuzzy_intent_score(
                predicted_intent,
                true_intent
            )
        )

        fuzzy_intent_match = (
            exact_intent_match
            or fuzzy_score >= FUZZY_MATCH_THRESHOLD
        )

        router_evaluation_records.append({
            "test_row_id": test_row_id,
            "instruction": query,
            "true_intent": true_intent,
            "true_category": true_category,
            "router_raw_output": (
                router_result["raw_output"]
            ),
            "router_parse_valid": (
                router_result["valid"]
            ),
            "strict_format_valid": (
                strict_result[
                    "strict_format_valid"
                ]
            ),
            "strict_format_error": (
                strict_result[
                    "strict_format_error"
                ]
            ),
            "predicted_intent": (
                predicted_intent
            ),
            "predicted_category": (
                predicted_category
            ),
            "exact_intent_match": (
                exact_intent_match
            ),
            "exact_category_match": (
                exact_category_match
            ),
            "fuzzy_intent_score": fuzzy_score,
            "fuzzy_intent_match": (
                fuzzy_intent_match
            ),
            "router_error": (
                router_result["error"]
            ),
            "router_execution_success": True,
            "execution_error": None
        })

    except Exception as error:

        router_evaluation_records.append({
            "test_row_id": test_row_id,
            "instruction": query,
            "true_intent": true_intent,
            "true_category": true_category,
            "router_raw_output": None,
            "router_parse_valid": False,
            "strict_format_valid": False,
            "strict_format_error": (
                "Router execution failed"
            ),
            "predicted_intent": None,
            "predicted_category": None,
            "exact_intent_match": False,
            "exact_category_match": False,
            "fuzzy_intent_score": 0.0,
            "fuzzy_intent_match": False,
            "router_error": None,
            "router_execution_success": False,
            "execution_error": (
                f"{type(error).__name__}: {error}"
            )
        })

    if (
        len(router_evaluation_records) % 50 == 0
        or len(router_evaluation_records) == len(df_test)
    ):
        pd.DataFrame(
            router_evaluation_records
        ).to_csv(
            ROUTER_CHECKPOINT_PATH,
            index=False
        )


v2_router_results_df = pd.DataFrame(
    router_evaluation_records
)

assert len(v2_router_results_df) == len(
    df_test
)

# Calculate complete held-out test metrics
v2_router_execution_success = (
    v2_router_results_df[
        "router_execution_success"
    ].mean()
    * 100
)

v2_format_adherence = (
    v2_router_results_df[
        "strict_format_valid"
    ].mean()
    * 100
)

v2_exact_intent_accuracy = (
    v2_router_results_df[
        "exact_intent_match"
    ].mean()
    * 100
)

v2_fuzzy_intent_accuracy = (
    v2_router_results_df[
        "fuzzy_intent_match"
    ].mean()
    * 100
)

v2_exact_category_accuracy = (
    v2_router_results_df[
        "exact_category_match"
    ].mean()
    * 100
)

Evaluating fine-tuned router:   0%|          | 0/400 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=

In [11]:
# Build the adversarial subset using regex filtering
ADVERSARIAL_REGEX = re.compile(
    r"\b("
    r"still|never|terrible|"
    r"frustrated|frustrating|"
    r"angry|upset|awful|"
    r"disappointed|disappointing"
    r")\b",
    flags=re.IGNORECASE
)

v2_router_results_df[
    "is_adversarial"
] = (
    v2_router_results_df[
        "instruction"
    ]
    .astype(str)
    .str.contains(
        ADVERSARIAL_REGEX,
        regex=True,
        na=False
    )
)

v2_adversarial_df = (
    v2_router_results_df[
        v2_router_results_df[
            "is_adversarial"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

if len(v2_adversarial_df) > 0:

    v2_adversarial_format_adherence = (
        v2_adversarial_df[
            "strict_format_valid"
        ].mean()
        * 100
    )

    v2_adversarial_exact_accuracy = (
        v2_adversarial_df[
            "exact_intent_match"
        ].mean()
        * 100
    )

    v2_adversarial_fuzzy_accuracy = (
        v2_adversarial_df[
            "fuzzy_intent_match"
        ].mean()
        * 100
    )

else:
    v2_adversarial_format_adherence = np.nan
    v2_adversarial_exact_accuracy = np.nan
    v2_adversarial_fuzzy_accuracy = np.nan

# Display router evaluation results
router_metric_summary = pd.DataFrame({
    "evaluation_subset": [
        "Complete held-out test split",
        "Regex-derived adversarial subset"
    ],
    "records": [
        len(v2_router_results_df),
        len(v2_adversarial_df)
    ],
    "format_adherence_percent": [
        v2_format_adherence,
        v2_adversarial_format_adherence
    ],
    "exact_intent_accuracy_percent": [
        v2_exact_intent_accuracy,
        v2_adversarial_exact_accuracy
    ],
    "fuzzy_intent_accuracy_percent": [
        v2_fuzzy_intent_accuracy,
        v2_adversarial_fuzzy_accuracy
    ]
})

print("Solution V2 router evaluation")
print("=" * 70)
print(
    f"Router execution success : "
    f"{v2_router_execution_success:.2f}%"
)
print(
    f"Strict format adherence  : "
    f"{v2_format_adherence:.2f}%"
)
print(
    f"Exact intent accuracy    : "
    f"{v2_exact_intent_accuracy:.2f}%"
)
print(
    f"Fuzzy intent accuracy    : "
    f"{v2_fuzzy_intent_accuracy:.2f}%"
)
print(
    f"Exact category accuracy  : "
    f"{v2_exact_category_accuracy:.2f}%"
)
print(
    f"Adversarial records      : "
    f"{len(v2_adversarial_df)}"
)
print(
    f"Router evaluation time   : "
    f"{(time.perf_counter() - router_evaluation_start) / 60:.2f} minutes"
)

display(
    router_metric_summary.round(2)
)


# Display incorrect router predictions
incorrect_router_predictions = (
    v2_router_results_df[
        ~v2_router_results_df[
            "exact_intent_match"
        ]
    ]
)

print("\nRepresentative router errors:")

if incorrect_router_predictions.empty:
    print(
        "No exact-match intent errors were detected."
    )
else:
    display(
        incorrect_router_predictions[
            [
                "instruction",
                "true_intent",
                "predicted_intent",
                "fuzzy_intent_score",
                "router_raw_output",
                "strict_format_valid"
            ]
        ].head(10)
    )

# Create the same 50-row synthesis sample as Notebook 5
if V2_EVAL_SAMPLE_SIZE > len(df_test):
    raise ValueError(
        "V2_EVAL_SAMPLE_SIZE exceeds the held-out test size."
    )

v2_synthesis_sample_df = (
    df_test
    .sample(
        n=V2_EVAL_SAMPLE_SIZE,
        random_state=RANDOM_STATE
    )
    .reset_index(drop=True)
)

v2_synthesis_sample_df[
    "evaluation_id"
] = np.arange(
    len(v2_synthesis_sample_df)
)


# Attach the existing full-test router results by instruction
router_lookup = (
    v2_router_results_df
    .set_index("instruction")
    .to_dict(orient="index")
)

# Generate oracle SOP-grounded references
def generate_oracle_reference(
    query,
    true_intent,
    true_category
):
    """
    Generate an SOP-grounded reference using the ground-truth
    intent and category.
    """

    oracle_router_result = {
        "valid": True,
        "intent": str(true_intent).strip().lower(),
        "category": str(true_category).strip().upper(),
        "error": None,
        "raw_output": None
    }

    oracle_retrieval = retrieve_v2_context(
        customer_query=query,
        router_result=oracle_router_result,
        k=REFERENCE_TOP_K
    )

    context_blocks = []

    for item in oracle_retrieval[
        "retrieved_items"
    ]:
        context_blocks.append(
            f"[Source: {item['source_file']} | "
            f"Chunk: {item['chunk_id']}]\n"
            f"{item['context']}"
        )

    combined_context = "\n\n".join(
        context_blocks
    )

    messages = [
        {
            "role": "system",
            "content": (
                "Create a concise reference answer for evaluating "
                "a customer-support system. Use only the supplied "
                "SOP context. Do not invent policies, timelines, "
                "procedures, departments, contact details, refunds, "
                "or guarantees. If the context is insufficient, "
                "state that the policy does not provide enough "
                "information."
            )
        },
        {
            "role": "user",
            "content": f"""
Customer query:
{str(query).strip()}

Ground-truth intent:
{str(true_intent).strip().lower()}

Ground-truth category:
{str(true_category).strip().upper()}

SOP context:
{combined_context}

Write the policy-grounded reference answer only.
""".strip()
        }
    ]

    reference_answer = generate_text(
        model=generator_model,
        device=GENERATOR_DEVICE,
        messages=messages,
        max_new_tokens=ANSWER_MAX_NEW_TOKENS
    )

    top_item = oracle_retrieval[
        "retrieved_items"
    ][0]

    return {
        "reference_output": reference_answer,
        "reference_source": (
            top_item["source_file"]
        ),
        "reference_chunk_id": (
            top_item["chunk_id"]
        ),
        "reference_context": (
            combined_context
        )
    }

# Generate Solution V2 answers and references
v2_synthesis_records = []
synthesis_start_time = time.perf_counter()

for _, row in tqdm(
    v2_synthesis_sample_df.iterrows(),
    total=len(v2_synthesis_sample_df),
    desc="Evaluating final Solution V2 synthesis"
):
    query = str(row["instruction"]).strip()

    router_row = router_lookup[query]

    router_result = {
        "raw_output": (
            router_row["router_raw_output"]
        ),
        "valid": bool(
            router_row["router_parse_valid"]
        ),
        "intent": (
            router_row["predicted_intent"]
        ),
        "category": (
            router_row["predicted_category"]
        ),
        "error": (
            router_row["router_error"]
        )
    }

    result_record = {
        "evaluation_id": int(
            row["evaluation_id"]
        ),
        "instruction": query,
        "true_intent": (
            str(row["intent"])
            .strip()
            .lower()
        ),
        "true_category": (
            str(row["category"])
            .strip()
            .upper()
        ),
        "router_raw_output": (
            router_result["raw_output"]
        ),
        "router_valid": (
            router_result["valid"]
        ),
        "predicted_intent": (
            router_result["intent"]
        ),
        "predicted_category": (
            router_result["category"]
        ),
        "final_answer": None,
        "retrieved_source": None,
        "retrieved_chunk_id": None,
        "retrieval_distance": None,
        "retrieved_context": None,
        "reference_output": None,
        "reference_source": None,
        "reference_chunk_id": None,
        "reference_context": None,
        "synthesis_success": False,
        "synthesis_error": None
    }

    try:
        retrieval_result = retrieve_v2_context(
            customer_query=query,
            router_result=router_result,
            k=V2_TOP_K
        )

        synthesis_result = synthesise_v2_answer(
            customer_query=query,
            router_result=router_result,
            retrieval_result=retrieval_result
        )

        top_retrieval = (
            retrieval_result[
                "retrieved_items"
            ][0]
        )

        reference_result = generate_oracle_reference(
            query=query,
            true_intent=row["intent"],
            true_category=row["category"]
        )

        result_record.update({
            "final_answer": (
                synthesis_result[
                    "final_answer"
                ]
            ),
            "retrieved_source": (
                top_retrieval[
                    "source_file"
                ]
            ),
            "retrieved_chunk_id": (
                top_retrieval[
                    "chunk_id"
                ]
            ),
            "retrieval_distance": float(
                top_retrieval[
                    "distance"
                ]
            ),
            "retrieved_context": (
                synthesis_result[
                    "combined_context"
                ]
            ),
            "reference_output": (
                reference_result[
                    "reference_output"
                ]
            ),
            "reference_source": (
                reference_result[
                    "reference_source"
                ]
            ),
            "reference_chunk_id": (
                reference_result[
                    "reference_chunk_id"
                ]
            ),
            "reference_context": (
                reference_result[
                    "reference_context"
                ]
            ),
            "synthesis_success": True
        })

    except Exception as error:
        result_record[
            "synthesis_error"
        ] = (
            f"{type(error).__name__}: {error}"
        )

    v2_synthesis_records.append(
        result_record
    )

    if (
        len(v2_synthesis_records) % 10 == 0
        or len(v2_synthesis_records)
        == len(v2_synthesis_sample_df)
    ):
        pd.DataFrame(
            v2_synthesis_records
        ).to_csv(
            SYNTHESIS_CHECKPOINT_PATH,
            index=False
        )


v2_synthesis_results_df = pd.DataFrame(
    v2_synthesis_records
)

successful_synthesis_count = int(
    v2_synthesis_results_df[
        "synthesis_success"
    ].sum()
)

assert successful_synthesis_count == len(
    v2_synthesis_results_df
), (
    "One or more Solution V2 synthesis records failed."
)

Solution V2 router evaluation
Router execution success : 100.00%
Strict format adherence  : 100.00%
Exact intent accuracy    : 73.25%
Fuzzy intent accuracy    : 84.75%
Exact category accuracy  : 75.75%
Adversarial records      : 0
Router evaluation time   : 6.19 minutes


/tmp/ipykernel_6919/4294815395.py:19: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(


,evaluation_subset,records,format_adherence_percent,exact_intent_accuracy_percent,fuzzy_intent_accuracy_percent
0,Complete held-out test split,400,100.0,73.25,84.75
1,Regex-derived adversarial subset,0,NaN,NaN,NaN



Representative router errors:


,instruction,true_intent,predicted_intent,fuzzy_intent_score,router_raw_output,strict_format_valid
2,I am waiting for a bloody compensation of {{Re...,track_refund,check_refund_amount,66.666667,"{""intent"":""check_refund_amount"",""category"":""RE...",True
13,I need help to check what shipping methods are...,delivery_options,get_shipping_options,66.666667,"{""intent"":""get_shipping_options"",""category"":""S...",True
14,I expect a compensation of {{Refund Amount}} d...,track_refund,request_refund,66.666667,"{""intent"":""request_refund"",""category"":""REFUND""}",True
22,wanna see order {{Order Number}} status acn ya...,track_order,check_order,72.727273,"{""intent"":""check_order"",""category"":""ORDER""}",True
23,how to see if there is anything wrong with the...,track_refund,check_complaint,22.222222,"{""intent"":""check_complaint"",""category"":""COMPLA...",True
27,I do not know how I can enter the new delivery...,set_up_shipping_address,change_shipping_address,80.000000,"{""intent"":""change_shipping_address"",""category""...",True
29,assistance informing of problems with payment,payment_issue,payment_problems,73.684211,"{""intent"":""payment_problems"",""category"":""PAYME...",True
35,is there a section to leave an opinion about a...,review,review_service,100.000000,"{""intent"":""review_service"",""category"":""FEEDBACK""}",True
42,I want help restoring my user account PIN,recover_password,recover_user_account_password,100.000000,"{""intent"":""recover_user_account_password"",""cat...",True
52,I can't check if there is anything wrong with ...,track_refund,check_refund_status,66.666667,"{""intent"":""check_refund_status"",""category"":""RE...",True


Evaluating final Solution V2 synthesis:   0%|          | 0/50 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:447: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info

In [23]:
# 11. Configure ROUGE and BLEU scoring
rouge_evaluator = rouge_scorer.RougeScorer(
    [
        "rouge1",
        "rougeL"
    ],
    use_stemmer=True
)

bleu_smoothing = (
    SmoothingFunction().method1
)


def calculate_generation_metrics(
    reference,
    prediction
):
    """Calculate ROUGE-1, ROUGE-L and sentence BLEU."""

    reference = str(reference).strip()
    prediction = str(prediction).strip()

    if not reference or not prediction:
        return {
            "rouge1": 0.0,
            "rougeL": 0.0,
            "bleu": 0.0
        }

    rouge_scores = rouge_evaluator.score(
        reference,
        prediction
    )

    bleu_score = sentence_bleu(
        [
            reference.lower().split()
        ],
        prediction.lower().split(),
        smoothing_function=bleu_smoothing
    )

    return {
        "rouge1": (
            rouge_scores[
                "rouge1"
            ].fmeasure
        ),
        "rougeL": (
            rouge_scores[
                "rougeL"
            ].fmeasure
        ),
        "bleu": bleu_score
    }

# 12. Calculate per-row synthesis metrics
v2_generation_metric_rows = []

for _, row in (
    v2_synthesis_results_df.iterrows()
):
    v2_generation_metric_rows.append(
        calculate_generation_metrics(
            reference=row[
                "reference_output"
            ],
            prediction=row[
                "final_answer"
            ]
        )
    )


v2_synthesis_results_df[
    "v2_rouge1"
] = [
    result["rouge1"]
    for result in v2_generation_metric_rows
]

v2_synthesis_results_df[
    "v2_rougeL"
] = [
    result["rougeL"]
    for result in v2_generation_metric_rows
]

v2_synthesis_results_df[
    "v2_bleu"
] = [
    result["bleu"]
    for result in v2_generation_metric_rows
]

v2_synthesis_results_df[
    "retrieval_source_match"
] = (
    v2_synthesis_results_df[
        "retrieved_source"
    ]
    .fillna("")
    .str.lower()
    .str.strip()
    ==
    v2_synthesis_results_df[
        "reference_source"
    ]
    .fillna("")
    .str.lower()
    .str.strip()
)

# 13. Aggregate final synthesis metrics
v2_mean_rouge1 = (
    v2_synthesis_results_df[
        "v2_rouge1"
    ].mean()
)

v2_mean_rougeL = (
    v2_synthesis_results_df[
        "v2_rougeL"
    ].mean()
)

v2_mean_bleu = (
    v2_synthesis_results_df[
        "v2_bleu"
    ].mean()
)

v2_retrieval_source_match_rate = (
    v2_synthesis_results_df[
        "retrieval_source_match"
    ].mean()
    * 100
)

v2_metric_summary = pd.DataFrame({
    "metric": [
        "Format Adherence",
        "Exact Intent Accuracy",
        "Fuzzy Intent Accuracy",
        "Adversarial Exact Accuracy",
        "Adversarial Fuzzy Accuracy",
        "ROUGE-1",
        "ROUGE-L",
        "BLEU",
        "Top-1 Source Agreement"
    ],
    "value": [
        v2_format_adherence,
        v2_exact_intent_accuracy,
        v2_fuzzy_intent_accuracy,
        v2_adversarial_exact_accuracy,
        v2_adversarial_fuzzy_accuracy,
        v2_mean_rouge1,
        v2_mean_rougeL,
        v2_mean_bleu,
        v2_retrieval_source_match_rate
    ],
    "unit": [
        "percent",
        "percent",
        "percent",
        "percent",
        "percent",
        "score",
        "score",
        "score",
        "percent"
    ]
})

# Display final Solution V2 evaluation
print("\nSolution V2 final-synthesis evaluation")
print("=" * 70)
print(
    f"Synthesis records          : "
    f"{len(v2_synthesis_results_df)}"
)
print(
    f"Successful synthesis       : "
    f"{successful_synthesis_count}"
)
print(
    f"Mean ROUGE-1              : "
    f"{v2_mean_rouge1:.4f}"
)
print(
    f"Mean ROUGE-L              : "
    f"{v2_mean_rougeL:.4f}"
)
print(
    f"Mean BLEU                 : "
    f"{v2_mean_bleu:.4f}"
)
print(
    f"Top-1 source agreement    : "
    f"{v2_retrieval_source_match_rate:.2f}%"
)
print(
    f"Synthesis evaluation time : "
    f"{(time.perf_counter() - synthesis_start_time) / 60:.2f} minutes"
)

display(
    v2_metric_summary.round(4)
)

print("\nRepresentative Solution V2 results:")

display(
    v2_synthesis_results_df[
        [
            "instruction",
            "true_intent",
            "predicted_intent",
            "router_valid",
            "reference_source",
            "retrieved_source",
            "reference_output",
            "final_answer",
            "v2_rouge1",
            "v2_rougeL",
            "v2_bleu"
        ]
    ].head()
)


Solution V2 final-synthesis evaluation
Synthesis records          : 50
Successful synthesis       : 50
Mean ROUGE-1              : 0.3285
Mean ROUGE-L              : 0.1890
Mean BLEU                 : 0.0338
Top-1 source agreement    : 84.00%
Synthesis evaluation time : 55.68 minutes


,metric,value,unit
0,Format Adherence,100.0000,percent
1,Exact Intent Accuracy,73.2500,percent
2,Fuzzy Intent Accuracy,84.7500,percent
3,Adversarial Exact Accuracy,77.9412,percent
4,Adversarial Fuzzy Accuracy,88.2353,percent
5,ROUGE-1,0.3285,score
6,ROUGE-L,0.1890,score
7,BLEU,0.0338,score
8,Top-1 Source Agreement,84.0000,percent



Representative Solution V2 results:


,instruction,true_intent,predicted_intent,router_valid,reference_source,retrieved_source,reference_output,final_answer,v2_rouge1,v2_rougeL,v2_bleu
0,I try to chat with an operator,contact_human_agent,contact_human_agent,True,escalation_matrix.md,escalation_matrix.md,When a customer attempts to initiate a convers...,The provided context indicates that your reque...,0.437500,0.281250,0.024117
1,can ya direct to me an assistant,contact_human_agent,contact_human_agent,True,escalation_matrix.md,escalation_matrix.md,When a customer queries about directing them t...,The provided context indicates that you can di...,0.137405,0.076336,0.001872
2,I need assistance editing my delivery address,change_shipping_address,change_shipping_address,True,order_tracking.md,order_tracking.md,### Evaluation Criteria for Customer Support S...,Thank you for reaching out to us. To assist yo...,0.318681,0.175824,0.003971
3,there is an issue trying to change my shipping...,change_shipping_address,change_shipping_address,True,order_tracking.md,order_tracking.md,### Evaluation of Customer Support System Resp...,The system has detected an issue while attempt...,0.414634,0.182927,0.010558
4,i dont knoq how i can report errors with payments,payment_issue,payment_issue,True,payment_methods.md,payment_methods.md,### Policy Grounding:\n\nWhen a customer repor...,"To report errors related to your payments, ple...",0.388571,0.285714,0.117168


In [22]:
# REBUILD A NON-EMPTY REGEX-DERIVED ADVERSARIAL SUBSET
EXPANDED_ADVERSARIAL_REGEX = re.compile(
    r"\b("
    r"still|never|terrible|frustrat(?:ed|ing)|angry|upset|awful|"
    r"disappoint(?:ed|ing)|issue|problem|error|wrong|failed|"
    r"cannot|can't|don't|not"
    r")\b",
    flags=re.IGNORECASE
)

v2_router_results_df["is_adversarial"] = (
    v2_router_results_df["instruction"]
    .astype(str)
    .str.contains(
        EXPANDED_ADVERSARIAL_REGEX,
        na=False
    )
)

v2_adversarial_df = (
    v2_router_results_df[
        v2_router_results_df["is_adversarial"]
    ]
    .copy()
    .reset_index(drop=True)
)

if v2_adversarial_df.empty:
    raise RuntimeError(
        "The expanded regex still produced no adversarial records."
    )

v2_adversarial_format_adherence = (
    v2_adversarial_df["strict_format_valid"].mean() * 100
)

v2_adversarial_exact_accuracy = (
    v2_adversarial_df["exact_intent_match"].mean() * 100
)

v2_adversarial_fuzzy_accuracy = (
    v2_adversarial_df["fuzzy_intent_match"].mean() * 100
)

print(f"Adversarial records         : {len(v2_adversarial_df)}")
print(
    f"Adversarial format adherence: "
    f"{v2_adversarial_format_adherence:.2f}%"
)
print(
    f"Adversarial exact accuracy  : "
    f"{v2_adversarial_exact_accuracy:.2f}%"
)
print(
    f"Adversarial fuzzy accuracy  : "
    f"{v2_adversarial_fuzzy_accuracy:.2f}%"
)

Adversarial records         : 68
Adversarial format adherence: 100.00%
Adversarial exact accuracy  : 77.94%
Adversarial fuzzy accuracy  : 88.24%


/tmp/ipykernel_6919/3229725143.py:14: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(


### Solution V2 Evaluation Interpretation

The fine-tuned intent router was evaluated on all **400 records** in the leakage-free held-out test split. It achieved strict JSON format adherence of **100.00%**, exact intent accuracy of **73.25%**, and fuzzy intent accuracy of **84.75%**. The higher fuzzy-match score shows that some incorrect exact predictions were nevertheless semantically close to the ground-truth intent label.

The expanded regex-derived adversarial subset contained **68 records**. On this subset, the fine-tuned router achieved an exact intent accuracy of **77.94%** and a fuzzy intent accuracy of **88.24%**.

Compared with the complete held-out test split, router performance improved slightly for queries containing negative, uncertain, or problem-oriented wording. Exact accuracy increased by 4.69 percentage points, while fuzzy accuracy increased by 3.49 percentage points. However, the subset is automatically derived through keyword filtering rather than manually constructed adversarial examples, so the result should be interpreted as a targeted robustness check rather than a comprehensive adversarial evaluation.

Final answer synthesis was evaluated on the same reproducible **50-row held-out sample** used for Solution V1. Solution V2 achieved mean scores of **0.3285** for ROUGE-1, **0.1890** for ROUGE-L, and **0.0338** for BLEU. The router-guided retriever selected the same top-ranked SOP as the oracle retrieval process in **84.00%** of cases.

These results evaluate separate stages of the Hybrid RAG system. The **100% JSON adherence** demonstrates reliable structured output, while the intent-accuracy results measure routing quality. The **84% top-1 source agreement** indicates that structured router predictions generally improved policy selection, although routing errors still caused incorrect retrieval in some cases. ROUGE and BLEU measure lexical similarity to SOP-grounded references and should be interpreted alongside routing and retrieval accuracy rather than as standalone measures of factual correctness.

#### **4.4.2 Analyse Fine-Tuning Impact [2 marks]**
**The Task:** Compare Solution V1 (Naive RAG) against Solution V2 (Hybrid RAG) to quantify the improvement attributable to fine-tuning.

**Hints & Tips:**
* Load `v1_metrics.csv` from Notebook 5 and compare against the V2 scores you just computed.
* Compute improvement percentages: `(v2 - v1) / v1 * 100` for each metric.
* Attribute the delta specifically to fine-tuning — retrieval was already present in V1, so any gain here is the router's contribution.

**Learner Inference:** This isolates fine-tuning's contribution, just as Task 3.4 isolated retrieval's — together they decompose the full system's improvement.

In [24]:
# YOUR CODE HERE
import numpy as np
import pandas as pd
from IPython.display import display

# Load Solution V1 per-row metrics from Notebook 5
V1_METRICS_PATH = (
    DRIVE_PROJECT_DIR
    / "artifacts"
    / "notebook_5"
    / "v1_metrics.csv"
)

if not V1_METRICS_PATH.exists():
    raise FileNotFoundError(
        f"Notebook 5 v1_metrics.csv was not found at:\n"
        f"{V1_METRICS_PATH}"
    )

v1_metrics_df = pd.read_csv(
    V1_METRICS_PATH
)

required_v1_columns = {
    "instruction",
    "naive_rag_success",
    "naive_rag_format_valid",
    "naive_rag_rouge1",
    "naive_rag_rougeL",
    "naive_rag_bleu",
    "retrieval_source_match",
    "retrieved_source",
    "reference_source"
}

missing_v1_columns = (
    required_v1_columns
    - set(v1_metrics_df.columns)
)

if missing_v1_columns:
    raise ValueError(
        f"v1_metrics.csv is missing columns: "
        f"{sorted(missing_v1_columns)}"
    )

print("Solution V1 metrics loaded")
print(f"V1 records: {len(v1_metrics_df):,}")

# Prepare matching keys for the V1 and V2 samples
def create_query_key(series):
    """Create a normalised query key for safe row matching."""

    return (
        series
        .astype(str)
        .str.lower()
        .str.strip()
        .str.replace(
            r"\s+",
            " ",
            regex=True
        )
    )


v1_comparison_source = (
    v1_metrics_df.copy()
)

v2_comparison_source = (
    v2_synthesis_results_df.copy()
)

v1_comparison_source["_query_key"] = (
    create_query_key(
        v1_comparison_source["instruction"]
    )
)

v2_comparison_source["_query_key"] = (
    create_query_key(
        v2_comparison_source["instruction"]
    )
)

assert v1_comparison_source[
    "_query_key"
].is_unique

assert v2_comparison_source[
    "_query_key"
].is_unique

v1_query_set = set(
    v1_comparison_source["_query_key"]
)

v2_query_set = set(
    v2_comparison_source["_query_key"]
)

assert v1_query_set == v2_query_set, (
    "Solution V1 and V2 were not evaluated on the same "
    "held-out queries."
)

# Select and rename comparable per-row metrics
v1_selected = (
    v1_comparison_source[
        [
            "_query_key",
            "instruction",
            "true_intent",
            "true_category",
            "naive_rag_output",
            "retrieved_source",
            "reference_source",
            "retrieval_source_match",
            "naive_rag_success",
            "naive_rag_format_valid",
            "naive_rag_rouge1",
            "naive_rag_rougeL",
            "naive_rag_bleu"
        ]
    ]
    .rename(
        columns={
            "instruction": "v1_instruction",
            "naive_rag_output": "v1_output",
            "retrieved_source": "v1_retrieved_source",
            "reference_source": "v1_reference_source",
            "retrieval_source_match": "v1_source_match",
            "naive_rag_success": "v1_synthesis_success",
            "naive_rag_format_valid": "v1_format_valid",
            "naive_rag_rouge1": "v1_rouge1",
            "naive_rag_rougeL": "v1_rougeL",
            "naive_rag_bleu": "v1_bleu"
        }
    )
)

v2_selected = (
    v2_comparison_source[
        [
            "_query_key",
            "instruction",
            "router_valid",
            "predicted_intent",
            "predicted_category",
            "final_answer",
            "retrieved_source",
            "reference_source",
            "retrieval_source_match",
            "synthesis_success",
            "v2_rouge1",
            "v2_rougeL",
            "v2_bleu"
        ]
    ]
    .rename(
        columns={
            "instruction": "v2_instruction",
            "final_answer": "v2_output",
            "retrieved_source": "v2_retrieved_source",
            "reference_source": "v2_reference_source",
            "retrieval_source_match": "v2_source_match",
            "synthesis_success": "v2_synthesis_success"
        }
    )
)

v1_v2_comparison_df = v1_selected.merge(
    v2_selected,
    on="_query_key",
    how="inner",
    validate="one_to_one"
)

assert len(v1_v2_comparison_df) == len(
    v2_synthesis_results_df
)

def normalise_source(series):
    return (
        series.fillna("")
        .astype(str)
        .str.lower()
        .str.strip()
    )

# Compare both systems with the same V2 oracle reference
common_reference_sources = normalise_source(
    v1_v2_comparison_df["v2_reference_source"]
)

v1_v2_comparison_df["v1_source_match"] = (
    normalise_source(
        v1_v2_comparison_df["v1_retrieved_source"]
    )
    == common_reference_sources
)

v1_v2_comparison_df["v2_source_match"] = (
    normalise_source(
        v1_v2_comparison_df["v2_retrieved_source"]
    )
    == common_reference_sources
)

# 4. Calculate comparable V2 final-answer format adherence
v1_v2_comparison_df["v2_format_valid"] = (
    v1_v2_comparison_df[
        "v2_synthesis_success"
    ].astype(bool)
    &
    v1_v2_comparison_df[
        "v2_output"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
)

Solution V1 metrics loaded
V1 records: 50


In [25]:
# Calculate per-row metric changes
v1_v2_comparison_df[
    "rouge1_change"
] = (
    v1_v2_comparison_df["v2_rouge1"]
    - v1_v2_comparison_df["v1_rouge1"]
)

v1_v2_comparison_df[
    "rougeL_change"
] = (
    v1_v2_comparison_df["v2_rougeL"]
    - v1_v2_comparison_df["v1_rougeL"]
)

v1_v2_comparison_df[
    "bleu_change"
] = (
    v1_v2_comparison_df["v2_bleu"]
    - v1_v2_comparison_df["v1_bleu"]
)

# Classify retrieval changes caused by the router
v1_source_match = (
    v1_v2_comparison_df[
        "v1_source_match"
    ].astype(bool)
)

v2_source_match = (
    v1_v2_comparison_df[
        "v2_source_match"
    ].astype(bool)
)

v1_v2_comparison_df[
    "fine_tuning_retrieval_effect"
] = np.select(
    [
        (~v1_source_match) & v2_source_match,
        v1_source_match & (~v2_source_match),
        v1_source_match & v2_source_match
    ],
    [
        "Router corrected retrieval",
        "Router degraded retrieval",
        "Both retrieved oracle source"
    ],
    default="Both missed oracle source"
)

# Helper for relative percentage improvement
def relative_improvement(
    v1_value,
    v2_value
):
    """Calculate (V2 - V1) / V1 × 100."""

    if np.isclose(v1_value, 0):
        return np.nan

    return (
        (v2_value - v1_value)
        / v1_value
        * 100
    )

# Calculate aggregate Solution V1 and V2 metrics
v1_format_adherence = (
    v1_v2_comparison_df[
        "v1_format_valid"
    ].astype(bool).mean()
    * 100
)

v2_final_format_adherence = (
    v1_v2_comparison_df[
        "v2_format_valid"
    ].astype(bool).mean()
    * 100
)

v1_synthesis_success = (
    v1_v2_comparison_df[
        "v1_synthesis_success"
    ].astype(bool).mean()
    * 100
)

v2_synthesis_success = (
    v1_v2_comparison_df[
        "v2_synthesis_success"
    ].astype(bool).mean()
    * 100
)

v1_source_agreement = (
    v1_v2_comparison_df[
        "v1_source_match"
    ].astype(bool).mean()
    * 100
)

v2_source_agreement = (
    v1_v2_comparison_df[
        "v2_source_match"
    ].astype(bool).mean()
    * 100
)

v1_mean_rouge1 = (
    v1_v2_comparison_df[
        "v1_rouge1"
    ].mean()
)

v2_mean_rouge1 = (
    v1_v2_comparison_df[
        "v2_rouge1"
    ].mean()
)

v1_mean_rougeL = (
    v1_v2_comparison_df[
        "v1_rougeL"
    ].mean()
)

v2_mean_rougeL = (
    v1_v2_comparison_df[
        "v2_rougeL"
    ].mean()
)

v1_mean_bleu = (
    v1_v2_comparison_df[
        "v1_bleu"
    ].mean()
)

v2_mean_bleu = (
    v1_v2_comparison_df[
        "v2_bleu"
    ].mean()
)

In [26]:
# Build the fine-tuning impact summary
fine_tuning_impact_summary = pd.DataFrame({
    "metric": [
        "Final-Answer Format Adherence",
        "Synthesis Execution Success",
        "Top-1 Source Agreement",
        "ROUGE-1",
        "ROUGE-L",
        "BLEU"
    ],
    "solution_v1": [
        v1_format_adherence,
        v1_synthesis_success,
        v1_source_agreement,
        v1_mean_rouge1,
        v1_mean_rougeL,
        v1_mean_bleu
    ],
    "solution_v2": [
        v2_final_format_adherence,
        v2_synthesis_success,
        v2_source_agreement,
        v2_mean_rouge1,
        v2_mean_rougeL,
        v2_mean_bleu
    ],
    "unit": [
        "percent",
        "percent",
        "percent",
        "score",
        "score",
        "score"
    ]
})

fine_tuning_impact_summary[
    "absolute_change"
] = (
    fine_tuning_impact_summary[
        "solution_v2"
    ]
    - fine_tuning_impact_summary[
        "solution_v1"
    ]
)

fine_tuning_impact_summary[
    "relative_improvement_percent"
] = fine_tuning_impact_summary.apply(
    lambda row: relative_improvement(
        row["solution_v1"],
        row["solution_v2"]
    ),
    axis=1
)

fine_tuning_impact_summary[
    "fine_tuning_effect"
] = np.select(
    [
        fine_tuning_impact_summary[
            "absolute_change"
        ] > 0,
        fine_tuning_impact_summary[
            "absolute_change"
        ] < 0
    ],
    [
        "Improved",
        "Declined"
    ],
    default="Unchanged"
)

# Summarise per-row retrieval impact
fine_tuning_retrieval_case_summary = (
    v1_v2_comparison_df[
        "fine_tuning_retrieval_effect"
    ]
    .value_counts()
    .reindex(
        [
            "Router corrected retrieval",
            "Router degraded retrieval",
            "Both retrieved oracle source",
            "Both missed oracle source"
        ],
        fill_value=0
    )
    .rename_axis("retrieval_case")
    .reset_index(name="record_count")
)

fine_tuning_retrieval_case_summary[
    "percentage"
] = (
    fine_tuning_retrieval_case_summary[
        "record_count"
    ]
    / len(v1_v2_comparison_df)
    * 100
)

# Report Solution V2 router-only metrics separately
router_only_metric_summary = pd.DataFrame({
    "metric": [
        "Strict JSON Format Adherence",
        "Exact Intent Accuracy",
        "Fuzzy Intent Accuracy",
        "Exact Category Accuracy"
    ],
    "solution_v2": [
        v2_format_adherence,
        v2_exact_intent_accuracy,
        v2_fuzzy_intent_accuracy,
        v2_exact_category_accuracy
    ],
    "comparison_note": [
        "V1 has no structured intent router",
        "V1 has no intent prediction stage",
        "V1 has no intent prediction stage",
        "V1 has no category prediction stage"
    ]
})

# Display aggregate results
print("Solution V1 versus Solution V2")
print("=" * 72)
print(
    f"Matched held-out records : "
    f"{len(v1_v2_comparison_df)}"
)
print(
    f"V1 top-1 source agreement: "
    f"{v1_source_agreement:.2f}%"
)
print(
    f"V2 top-1 source agreement: "
    f"{v2_source_agreement:.2f}%"
)

display(
    fine_tuning_impact_summary.round({
        "solution_v1": 4,
        "solution_v2": 4,
        "absolute_change": 4,
        "relative_improvement_percent": 2
    })
)

print("\nPer-row retrieval impact attributable to the router:")

display(
    fine_tuning_retrieval_case_summary.round({
        "percentage": 2
    })
)

print("\nSolution V2 router-only metrics:")

display(
    router_only_metric_summary.round({
        "solution_v2": 2
    })
)

Solution V1 versus Solution V2
Matched held-out records : 50
V1 top-1 source agreement: 44.00%
V2 top-1 source agreement: 84.00%


,metric,solution_v1,solution_v2,unit,absolute_change,relative_improvement_percent,fine_tuning_effect
0,Final-Answer Format Adherence,100.0000,100.0000,percent,0.0000,0.00,Unchanged
1,Synthesis Execution Success,100.0000,100.0000,percent,0.0000,0.00,Unchanged
2,Top-1 Source Agreement,44.0000,84.0000,percent,40.0000,90.91,Improved
3,ROUGE-1,0.3155,0.3285,score,0.0131,4.14,Improved
4,ROUGE-L,0.1805,0.1890,score,0.0086,4.75,Improved
5,BLEU,0.0334,0.0338,score,0.0004,1.25,Improved



Per-row retrieval impact attributable to the router:


,retrieval_case,record_count,percentage
0,Router corrected retrieval,20,40.0
1,Router degraded retrieval,0,0.0
2,Both retrieved oracle source,22,44.0
3,Both missed oracle source,8,16.0



Solution V2 router-only metrics:


,metric,solution_v2,comparison_note
0,Strict JSON Format Adherence,100.00,V1 has no structured intent router
1,Exact Intent Accuracy,73.25,V1 has no intent prediction stage
2,Fuzzy Intent Accuracy,84.75,V1 has no intent prediction stage
3,Exact Category Accuracy,75.75,V1 has no category prediction stage


In [27]:
# Display examples of improved and degraded retrieval
print("\nExamples where fine-tuning corrected retrieval:")

corrected_examples = (
    v1_v2_comparison_df[
        v1_v2_comparison_df[
            "fine_tuning_retrieval_effect"
        ] == "Router corrected retrieval"
    ]
    .sort_values(
        "rouge1_change",
        ascending=False
    )
)

if corrected_examples.empty:
    print("No corrected retrieval cases were found.")
else:
    display(
        corrected_examples[
            [
                "v1_instruction",
                "true_intent",
                "predicted_intent",
                "v1_retrieved_source",
                "v2_retrieved_source",
                "v2_reference_source",
                "rouge1_change",
                "rougeL_change",
                "bleu_change"
            ]
        ].head(5)
    )


print("\nExamples where fine-tuning degraded retrieval:")

degraded_examples = (
    v1_v2_comparison_df[
        v1_v2_comparison_df[
            "fine_tuning_retrieval_effect"
        ] == "Router degraded retrieval"
    ]
    .sort_values(
        "rouge1_change",
        ascending=True
    )
)

if degraded_examples.empty:
    print("No degraded retrieval cases were found.")
else:
    display(
        degraded_examples[
            [
                "v1_instruction",
                "true_intent",
                "predicted_intent",
                "v1_retrieved_source",
                "v2_retrieved_source",
                "v2_reference_source",
                "rouge1_change",
                "rougeL_change",
                "bleu_change"
            ]
        ].head(5)
    )


Examples where fine-tuning corrected retrieval:


,v1_instruction,true_intent,predicted_intent,v1_retrieved_source,v2_retrieved_source,v2_reference_source,rouge1_change,rougeL_change,bleu_change
6,could you help me to acquire something?,place_order,place_order,technical_troubleshooting.md,billing_disputes.md,billing_disputes.md,0.378810,0.209332,0.087372
9,can you help me sending my feedback for your s...,review,review,working_hours.md,technical_troubleshooting.md,technical_troubleshooting.md,0.247884,0.100516,0.024949
0,I try to chat with an operator,contact_human_agent,contact_human_agent,working_hours.md,escalation_matrix.md,escalation_matrix.md,0.155147,0.163603,0.018296
32,help me sending some fucking feedback about yo...,review,review,data_privacy.md,technical_troubleshooting.md,technical_troubleshooting.md,0.147516,0.048137,0.012627
43,"I need to speak with customer support, help me",contact_customer_service,contact_customer_service,technical_troubleshooting.md,working_hours.md,working_hours.md,0.098873,0.091312,0.001146



Examples where fine-tuning degraded retrieval:
No degraded retrieval cases were found.


### Fine-Tuning Impact Interpretation

Solution V1 and Solution V2 were compared on the same **50 held-out customer queries**, ensuring that the observed differences are attributable to the fine-tuned intent router rather than to a different evaluation sample.

Using the same SOP-grounded oracle reference for both systems, top-1 source agreement increased from **44.00%** for Naive RAG to **84.00%** for Hybrid RAG. This represents an improvement of **40 percentage points** and a **90.91% relative increase**. Because retrieval was already present in Solution V1, this improvement reflects the contribution of fine-tuned intent classification and structured query construction.

Final-answer format adherence and synthesis execution success remained unchanged at **100.00%** for both solutions. ROUGE-1 increased from **0.3155** to **0.3285**, representing a **4.14% relative improvement**. ROUGE-L increased from **0.1805** to **0.1890**, a **4.75% relative improvement**, while BLEU increased from **0.0334** to **0.0338**, a smaller **1.25% relative improvement**. These results show that the router produced modest improvements in the final answer’s lexical similarity to the SOP-grounded references.

At the structural level, the fine-tuned router achieved **100.00% strict JSON adherence**, **73.25% exact intent accuracy**, **84.75% fuzzy intent accuracy**, and **75.75% exact category accuracy**. These metrics have no direct Solution V1 equivalent because Naive RAG does not contain a structured intent-classification stage.

At the individual-record level, the router corrected an incorrect Naive RAG retrieval in **20 cases (40%)** and did not degrade any previously correct retrievals in this evaluation sample. Both systems retrieved the oracle source in **22 cases (44%)**, while both missed it in **8 cases (16%)**.

Overall, fine-tuning produced a substantial improvement in retrieval precision. The remaining errors occurred where both systems failed to retrieve the oracle source, indicating that intent classification and the intent-to-policy search mapping remain areas for further improvement.

### **Task 4.5: Perform Comparative Analysis**

> Subtasks 4.5.1 (Compare All Versions) and 4.5.2 (Document Findings) are written up in the **Comparative Analysis Report PDF**. The cell below generates the scoring tables that feed that report.

**The Task:** Run all three architectures (Baseline, Naive RAG, Hybrid RAG) across the full held-out test split with SOP-grounded references, then export the per-row and summary CSVs.

**Hints & Tips:**
* SOP-grounded references reward policy-specific answers, ensuring Hybrid scores highest.
* This is the most compute-intensive cell — expect 15–30 min on T4. Use `df_test.head(50)` if time-constrained.
* Export `Comparative_Results_Full.csv` (per-row) and `Comparative_Results_Summary.csv` (aggregate).

In [28]:
# YOUR CODE HERE
import re
import numpy as np
import pandas as pd

from nltk.translate.bleu_score import (
    SmoothingFunction,
    sentence_bleu
)
from rouge_score import rouge_scorer
from tqdm.auto import tqdm
from IPython.display import display

# Load Solution V1 results
V1_METRICS_PATH = (
    DRIVE_PROJECT_DIR
    / "artifacts"
    / "notebook_5"
    / "v1_metrics.csv"
)

if not V1_METRICS_PATH.exists():
    raise FileNotFoundError(
        f"Solution V1 metrics were not found at:\n"
        f"{V1_METRICS_PATH}"
    )

v1_metrics_final_df = pd.read_csv(
    V1_METRICS_PATH
)

print("Loaded Solution V1 metrics")
print(f"Records: {len(v1_metrics_final_df):,}")

# Create a common query-matching key
def create_query_key(series):
    """Create a stable matching key from customer queries."""

    return (
        series
        .astype(str)
        .str.lower()
        .str.strip()
        .str.replace(
            r"\s+",
            " ",
            regex=True
        )
    )


v1_metrics_final_df["_query_key"] = (
    create_query_key(
        v1_metrics_final_df["instruction"]
    )
)

v2_synthesis_final_df = (
    v2_synthesis_results_df.copy()
)

v2_synthesis_final_df["_query_key"] = (
    create_query_key(
        v2_synthesis_final_df["instruction"]
    )
)

v2_router_final_df = (
    v2_router_results_df.copy()
)

v2_router_final_df["_query_key"] = (
    create_query_key(
        v2_router_final_df["instruction"]
    )
)

assert v1_metrics_final_df["_query_key"].is_unique
assert v2_synthesis_final_df["_query_key"].is_unique
assert v2_router_final_df["_query_key"].is_unique

# Select and rename architecture-specific columns
v1_selected = (
    v1_metrics_final_df[
        [
            "_query_key",
            "evaluation_id",
            "instruction",
            "true_intent",
            "true_category",

            "baseline_output",
            "baseline_success",
            "baseline_format_valid",
            "baseline_latency_seconds",
            "baseline_consistency_rate_percent",

            "naive_rag_output",
            "naive_rag_success",
            "naive_rag_format_valid",
            "naive_rag_latency_seconds",
            "naive_rag_consistency_rate_percent",

            "retrieved_source",
            "retrieved_chunk_id",
            "retrieval_distance",
            "retrieval_source_match"
        ]
    ]
    .rename(
        columns={
            "instruction": "customer_query",

            "naive_rag_output": "v1_output",
            "naive_rag_success": "v1_success",
            "naive_rag_format_valid": "v1_format_valid",
            "naive_rag_latency_seconds": "v1_latency_seconds",
            "naive_rag_consistency_rate_percent":
                "v1_consistency_rate_percent",

            "retrieved_source": "v1_retrieved_source",
            "retrieved_chunk_id": "v1_retrieved_chunk_id",
            "retrieval_distance": "v1_retrieval_distance",
            "retrieval_source_match": "v1_source_match"
        }
    )
)


v2_selected = (
    v2_synthesis_final_df[
        [
            "_query_key",
            "final_answer",
            "synthesis_success",
            "synthesis_error",

            "retrieved_source",
            "retrieved_chunk_id",
            "retrieval_distance",
            "retrieved_context",

            "reference_output",
            "reference_source",
            "reference_chunk_id",
            "reference_context",

            "retrieval_source_match"
        ]
    ]
    .rename(
        columns={
            "final_answer": "v2_output",
            "synthesis_success": "v2_success",
            "synthesis_error": "v2_synthesis_error",

            "retrieved_source": "v2_retrieved_source",
            "retrieved_chunk_id": "v2_retrieved_chunk_id",
            "retrieval_distance": "v2_retrieval_distance",
            "retrieved_context": "v2_retrieved_context",

            "retrieval_source_match": "v2_source_match"
        }
    )
)


router_selected = (
    v2_router_final_df[
        [
            "_query_key",
            "router_raw_output",
            "router_parse_valid",
            "strict_format_valid",
            "predicted_intent",
            "predicted_category",
            "exact_intent_match",
            "exact_category_match",
            "fuzzy_intent_score",
            "fuzzy_intent_match",
            "is_adversarial"
        ]
    ]
    .rename(
        columns={
            "router_parse_valid": "v2_router_parse_valid",
            "strict_format_valid": "v2_router_strict_format_valid",
            "predicted_intent": "v2_predicted_intent",
            "predicted_category": "v2_predicted_category",
            "exact_intent_match": "v2_exact_intent_match",
            "exact_category_match": "v2_exact_category_match",
            "fuzzy_intent_score": "v2_fuzzy_intent_score",
            "fuzzy_intent_match": "v2_fuzzy_intent_match"
        }
    )
)

# Merge the three system results
comparative_working_df = (
    v1_selected
    .merge(
        v2_selected,
        on="_query_key",
        how="inner",
        validate="one_to_one"
    )
    .merge(
        router_selected,
        on="_query_key",
        how="left",
        validate="one_to_one"
    )
)

assert len(comparative_working_df) == len(
    v2_synthesis_results_df
)

print(
    f"Matched comparison records: "
    f"{len(comparative_working_df):,}"
)

common_reference_sources = normalise_source(
    comparative_working_df["reference_source"]
)

comparative_working_df["v1_source_match"] = (
    normalise_source(
        comparative_working_df["v1_retrieved_source"]
    )
    == common_reference_sources
)

comparative_working_df["v2_source_match"] = (
    normalise_source(
        comparative_working_df["v2_retrieved_source"]
    )
    == common_reference_sources
)

Loaded Solution V1 metrics
Records: 50
Matched comparison records: 50


In [29]:
# Normalise Boolean columns loaded from CSV
def to_boolean(series):
    """Safely convert saved Boolean columns."""

    return (
        series
        .fillna(False)
        .map(
            lambda value:
                value
                if isinstance(value, bool)
                else str(value).strip().lower()
                in {"true", "1", "yes"}
        )
        .astype(bool)
    )


BOOLEAN_COLUMNS = [
    "baseline_success",
    "baseline_format_valid",
    "v1_success",
    "v1_format_valid",
    "v1_source_match",
    "v2_success",
    "v2_source_match",
    "v2_router_parse_valid",
    "v2_router_strict_format_valid",
    "v2_exact_intent_match",
    "v2_exact_category_match",
    "v2_fuzzy_intent_match",
    "is_adversarial"
]

for column in BOOLEAN_COLUMNS:
    comparative_working_df[column] = (
        to_boolean(
            comparative_working_df[column]
        )
    )


comparative_working_df["v2_format_valid"] = (
    comparative_working_df["v2_success"]
    &
    comparative_working_df[
        "v2_output"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
)

# Retrieve the exact V1 context using the saved chunk ID
def load_chroma_chunk(
    chunk_id,
    fallback_query
):
    """
    Load the exact chunk used by Solution V1.

    Falls back to raw-query retrieval only if the stored chunk ID
    cannot be located.
    """

    chunk_id = str(chunk_id).strip()

    stored_result = chroma_collection.get(
        ids=[chunk_id],
        include=[
            "documents",
            "metadatas"
        ]
    )

    stored_documents = (
        stored_result.get("documents")
        or []
    )

    stored_metadatas = (
        stored_result.get("metadatas")
        or []
    )

    if stored_documents:
        metadata = (
            stored_metadatas[0]
            if stored_metadatas
            else {}
        )

        return {
            "context": stored_documents[0],
            "source_file": metadata.get(
                "source_file",
                "unknown"
            ),
            "chunk_id": chunk_id
        }

    # Raw-query fallback
    query_embedding = embedding_model.encode(
        [str(fallback_query).strip()],
        normalize_embeddings=True
    ).astype(np.float32)

    fallback_result = chroma_collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=1,
        include=[
            "documents",
            "metadatas"
        ]
    )

    if not fallback_result["documents"][0]:
        raise RuntimeError(
            "Unable to recover the Solution V1 context."
        )

    fallback_metadata = (
        fallback_result["metadatas"][0][0]
    )

    return {
        "context": (
            fallback_result["documents"][0][0]
        ),
        "source_file": fallback_metadata.get(
            "source_file",
            "unknown"
        ),
        "chunk_id": fallback_metadata.get(
            "chunk_id",
            "unknown"
        )
    }


v1_context_records = []

for _, row in tqdm(
    comparative_working_df.iterrows(),
    total=len(comparative_working_df),
    desc="Recovering Solution V1 contexts"
):
    context_result = load_chroma_chunk(
        chunk_id=row["v1_retrieved_chunk_id"],
        fallback_query=row["customer_query"]
    )

    v1_context_records.append(
        context_result
    )


comparative_working_df[
    "v1_retrieved_context"
] = [
    result["context"]
    for result in v1_context_records
]

# Configure common ROUGE and BLEU scoring
common_rouge_scorer = rouge_scorer.RougeScorer(
    [
        "rouge1",
        "rougeL"
    ],
    use_stemmer=True
)

common_bleu_smoothing = (
    SmoothingFunction().method1
)


def calculate_common_metrics(
    reference,
    prediction
):
    """Calculate all metrics using one common reference."""

    reference = str(reference).strip()
    prediction = str(prediction).strip()

    if not reference or not prediction:
        return {
            "rouge1": 0.0,
            "rougeL": 0.0,
            "bleu": 0.0
        }

    rouge_scores = common_rouge_scorer.score(
        reference,
        prediction
    )

    bleu_value = sentence_bleu(
        [
            reference.lower().split()
        ],
        prediction.lower().split(),
        smoothing_function=common_bleu_smoothing
    )

    return {
        "rouge1": (
            rouge_scores["rouge1"].fmeasure
        ),
        "rougeL": (
            rouge_scores["rougeL"].fmeasure
        ),
        "bleu": bleu_value
    }

# Recalculate all three systems against identical references
for system_name, output_column in [
    ("baseline", "baseline_output"),
    ("v1", "v1_output"),
    ("v2", "v2_output")
]:
    metric_rows = [
        calculate_common_metrics(
            reference=row["reference_output"],
            prediction=row[output_column]
        )
        for _, row in (
            comparative_working_df.iterrows()
        )
    ]

    comparative_working_df[
        f"{system_name}_rouge1"
    ] = [
        result["rouge1"]
        for result in metric_rows
    ]

    comparative_working_df[
        f"{system_name}_rougeL"
    ] = [
        result["rougeL"]
        for result in metric_rows
    ]

    comparative_working_df[
        f"{system_name}_bleu"
    ] = [
        result["bleu"]
        for result in metric_rows
    ]

Recovering Solution V1 contexts:   0%|          | 0/50 [00:00<?, ?it/s]

In [30]:
# Common hallucination and functionality evaluator
CLAIM_SIMILARITY_THRESHOLD = 0.30
MISSING_FUNCTIONALITY_THRESHOLD = 0.25

POLICY_TERMS = [
    "refund",
    "replacement",
    "compensation",
    "credit",
    "cancel",
    "escalat",
    "contact",
    "email",
    "investigation",
    "eligible",
    "approved",
    "guarantee",
    "delivery",
    "shipping",
    "tracking",
    "password",
    "account",
    "payment",
    "charge",
    "fee",
    "business day"
]

ABSOLUTE_PROMISE_PATTERNS = [
    r"\bguaranteed?\b",
    r"\bdefinitely\b",
    r"\bautomatically\b",
    r"\bimmediately\b",
    r"\balways\b",
    r"\bwill definitely\b",
    r"\bwill be refunded\b",
    r"\bwill receive\b"
]

NUMERIC_CLAIM_PATTERN = re.compile(
    r"\b\d+(?:\.\d+)?"
    r"(?:\s*(?:-|–|—|to)\s*\d+(?:\.\d+)?)?"
    r"\s*(?:business\s+)?"
    r"(?:minutes?|hours?|days?|weeks?|months?|years?|"
    r"percent|%|dollars?|usd|rupees?|rs\.?)?\b",
    flags=re.IGNORECASE
)


def normalise_text(text):
    return re.sub(
        r"\s+",
        " ",
        str(text)
        .lower()
        .replace("–", "-")
        .replace("—", "-")
    ).strip()


def split_output_sentences(text):
    return [
        sentence.strip()
        for sentence in re.split(
            r"(?<=[.!?])\s+|\n+",
            str(text).strip()
        )
        if sentence.strip()
    ]


def semantic_similarity(
    first_text,
    second_text
):
    if (
        not str(first_text).strip()
        or not str(second_text).strip()
    ):
        return 0.0

    vectors = embedding_model.encode(
        [
            str(first_text).strip(),
            str(second_text).strip()
        ],
        normalize_embeddings=True
    )

    return float(
        np.dot(
            vectors[0],
            vectors[1]
        )
    )


def maximum_context_support(
    sentence,
    context_segments
):
    if not sentence or not context_segments:
        return 0.0

    vectors = embedding_model.encode(
        [sentence] + context_segments,
        normalize_embeddings=True
    )

    return float(
        (
            vectors[1:]
            @ vectors[0]
        ).max()
    )


def audit_comparative_output(
    output,
    support_context,
    reference_output
):
    """
    Apply the same transparent audit criteria to all systems.
    """

    output = (
        ""
        if pd.isna(output)
        else str(output).strip()
    )

    support_context = (
        ""
        if pd.isna(support_context)
        else str(support_context).strip()
    )

    reference_output = (
        ""
        if pd.isna(reference_output)
        else str(reference_output).strip()
    )

    if not output:
        return {
            "hallucination_detected": True,
            "functionally_correct": False,
            "missing_functionality": True,
            "reference_similarity": 0.0,
            "hallucination_reasons":
                "Empty or missing response"
        }

    normalised_context = normalise_text(
        support_context
    )

    context_segments = [
        segment.strip()
        for segment in re.split(
            r"\n\s*\n",
            support_context
        )
        if segment.strip()
    ]

    unsupported_numeric_claims = []

    for match in NUMERIC_CLAIM_PATTERN.finditer(
        output
    ):
        numeric_claim = (
            match.group(0).strip()
        )

        if (
            normalise_text(numeric_claim)
            not in normalised_context
        ):
            unsupported_numeric_claims.append(
                numeric_claim
            )

    unsupported_policy_sentences = []

    for sentence in split_output_sentences(
        output
    ):
        normalised_sentence = normalise_text(
            sentence
        )

        is_policy_claim = (
            any(
                term in normalised_sentence
                for term in POLICY_TERMS
            )
            or bool(
                NUMERIC_CLAIM_PATTERN.search(
                    sentence
                )
            )
        )

        if not is_policy_claim:
            continue

        support_score = maximum_context_support(
            sentence,
            context_segments
        )

        if support_score < CLAIM_SIMILARITY_THRESHOLD:
            unsupported_policy_sentences.append(
                sentence
            )

    unsupported_promises = []

    for pattern in ABSOLUTE_PROMISE_PATTERNS:
        for match in re.findall(
            pattern,
            output,
            flags=re.IGNORECASE
        ):
            phrase = normalise_text(match)

            if phrase not in normalised_context:
                unsupported_promises.append(
                    phrase
                )

    reference_similarity = semantic_similarity(
        output,
        reference_output
    )

    missing_functionality = (
        reference_similarity
        < MISSING_FUNCTIONALITY_THRESHOLD
    )

    hallucination_reasons = []

    if unsupported_numeric_claims:
        hallucination_reasons.append(
            "Unsupported numeric or timeline claim"
        )

    if unsupported_policy_sentences:
        hallucination_reasons.append(
            "Policy claim weakly supported by context"
        )

    if unsupported_promises:
        hallucination_reasons.append(
            "Unsupported absolute promise"
        )

    if missing_functionality:
        hallucination_reasons.append(
            "Missing required policy functionality"
        )

    hallucination_detected = bool(
        hallucination_reasons
    )

    return {
        "hallucination_detected":
            hallucination_detected,

        "functionally_correct":
            not hallucination_detected,

        "missing_functionality":
            missing_functionality,

        "reference_similarity":
            reference_similarity,

        "hallucination_reasons":
            " || ".join(
                hallucination_reasons
            )
    }

# Audit all three systems using the same methodology
SYSTEM_AUDIT_CONFIGURATION = {
    "baseline": {
        "output_column": "baseline_output",
        "context_column": "reference_context"
    },
    "v1": {
        "output_column": "v1_output",
        "context_column": "v1_retrieved_context"
    },
    "v2": {
        "output_column": "v2_output",
        "context_column": "v2_retrieved_context"
    }
}


for system_name, system_config in (
    SYSTEM_AUDIT_CONFIGURATION.items()
):
    audit_rows = []

    for _, row in tqdm(
        comparative_working_df.iterrows(),
        total=len(comparative_working_df),
        desc=f"Auditing {system_name}"
    ):
        audit_rows.append(
            audit_comparative_output(
                output=row[
                    system_config[
                        "output_column"
                    ]
                ],
                support_context=row[
                    system_config[
                        "context_column"
                    ]
                ],
                reference_output=row[
                    "reference_output"
                ]
            )
        )

    audit_df = (
        pd.DataFrame(audit_rows)
        .add_prefix(
            f"{system_name}_"
        )
    )

    comparative_working_df = pd.concat(
        [
            comparative_working_df.reset_index(
                drop=True
            ),
            audit_df.reset_index(
                drop=True
            )
        ],
        axis=1
    )

# Add retrieval-impact classifications
comparative_working_df[
    "v1_to_v2_retrieval_effect"
] = np.select(
    [
        (
            ~comparative_working_df[
                "v1_source_match"
            ]
            &
            comparative_working_df[
                "v2_source_match"
            ]
        ),
        (
            comparative_working_df[
                "v1_source_match"
            ]
            &
            ~comparative_working_df[
                "v2_source_match"
            ]
        ),
        (
            comparative_working_df[
                "v1_source_match"
            ]
            &
            comparative_working_df[
                "v2_source_match"
            ]
        )
    ],
    [
        "Router corrected retrieval",
        "Router degraded retrieval",
        "Both retrieved oracle source"
    ],
    default="Both missed oracle source"
)

# Create final per-row export table
COMPARATIVE_FULL_COLUMNS = [
    "evaluation_id",
    "customer_query",
    "true_intent",
    "true_category",

    "reference_output",
    "reference_source",
    "reference_chunk_id",

    # Baseline
    "baseline_output",
    "baseline_success",
    "baseline_format_valid",
    "baseline_latency_seconds",
    "baseline_rouge1",
    "baseline_rougeL",
    "baseline_bleu",
    "baseline_reference_similarity",
    "baseline_functionally_correct",
    "baseline_hallucination_detected",
    "baseline_missing_functionality",
    "baseline_hallucination_reasons",

    # Solution V1
    "v1_output",
    "v1_success",
    "v1_format_valid",
    "v1_latency_seconds",
    "v1_retrieved_source",
    "v1_retrieved_chunk_id",
    "v1_retrieval_distance",
    "v1_source_match",
    "v1_rouge1",
    "v1_rougeL",
    "v1_bleu",
    "v1_reference_similarity",
    "v1_functionally_correct",
    "v1_hallucination_detected",
    "v1_missing_functionality",
    "v1_hallucination_reasons",

    # Solution V2 router
    "router_raw_output",
    "v2_router_parse_valid",
    "v2_router_strict_format_valid",
    "v2_predicted_intent",
    "v2_predicted_category",
    "v2_exact_intent_match",
    "v2_exact_category_match",
    "v2_fuzzy_intent_score",
    "v2_fuzzy_intent_match",
    "is_adversarial",

    # Solution V2 final answer
    "v2_output",
    "v2_success",
    "v2_format_valid",
    "v2_retrieved_source",
    "v2_retrieved_chunk_id",
    "v2_retrieval_distance",
    "v2_source_match",
    "v2_rouge1",
    "v2_rougeL",
    "v2_bleu",
    "v2_reference_similarity",
    "v2_functionally_correct",
    "v2_hallucination_detected",
    "v2_missing_functionality",
    "v2_hallucination_reasons",

    # Comparative impact
    "v1_to_v2_retrieval_effect"
]

missing_comparative_columns = [
    column
    for column in COMPARATIVE_FULL_COLUMNS
    if column not in comparative_working_df.columns
]

if missing_comparative_columns:
    raise ValueError(
        "Missing comparative columns:\n"
        f"{missing_comparative_columns}"
    )

comparative_results_full_df = (
    comparative_working_df[
        COMPARATIVE_FULL_COLUMNS
    ]
    .copy()
    .reset_index(drop=True)
)

assert len(comparative_results_full_df) == 50
assert (
    comparative_results_full_df[
        "customer_query"
    ].is_unique
)

print("\nFull per-row comparative table created")
print("=" * 70)
print(
    f"Comparison records : "
    f"{len(comparative_results_full_df)}"
)
print(
    f"Comparison columns : "
    f"{len(comparative_results_full_df.columns)}"
)

display(
    comparative_results_full_df.head()
)

Auditing baseline:   0%|          | 0/50 [00:00<?, ?it/s]

Auditing v1:   0%|          | 0/50 [00:00<?, ?it/s]

Auditing v2:   0%|          | 0/50 [00:00<?, ?it/s]


Full per-row comparative table created
Comparison records : 50
Comparison columns : 61


,evaluation_id,customer_query,true_intent,true_category,reference_output,reference_source,reference_chunk_id,baseline_output,baseline_success,baseline_format_valid,...,v2_source_match,v2_rouge1,v2_rougeL,v2_bleu,v2_reference_similarity,v2_functionally_correct,v2_hallucination_detected,v2_missing_functionality,v2_hallucination_reasons,v1_to_v2_retrieval_effect
0,0,I try to chat with an operator,contact_human_agent,CONTACT,When a customer attempts to initiate a convers...,escalation_matrix.md,escalation_matrix_chunk_000,Hello! I'm here to help you. How can I assist ...,True,True,...,True,0.437500,0.281250,0.024117,0.738768,True,False,False,,Router corrected retrieval
1,1,can ya direct to me an assistant,contact_human_agent,CONTACT,When a customer queries about directing them t...,escalation_matrix.md,escalation_matrix_chunk_000,"I'm sorry, but I can't provide that service di...",True,True,...,True,0.137405,0.076336,0.001872,0.507330,True,False,False,,Router corrected retrieval
2,2,I need assistance editing my delivery address,change_shipping_address,SHIPPING,### Evaluation Criteria for Customer Support S...,order_tracking.md,order_tracking_chunk_004,Hello! I'm here to help you with your delivery...,True,True,...,True,0.318681,0.175824,0.003971,0.632299,False,True,False,Unsupported numeric or timeline claim || Polic...,Both retrieved oracle source
3,3,there is an issue trying to change my shipping...,change_shipping_address,SHIPPING,### Evaluation of Customer Support System Resp...,order_tracking.md,order_tracking_chunk_004,I'm sorry to hear that you're experiencing iss...,True,True,...,True,0.414634,0.182927,0.010558,0.716331,False,True,False,Unsupported numeric or timeline claim || Polic...,Both retrieved oracle source
4,4,i dont knoq how i can report errors with payments,payment_issue,PAYMENT,### Policy Grounding:\n\nWhen a customer repor...,payment_methods.md,payment_methods_chunk_004,"To report an error on your payment, you should...",True,True,...,True,0.388571,0.285714,0.117168,0.696630,False,True,False,Policy claim weakly supported by context,Router corrected retrieval


In [31]:
# YOUR CODE HERE
import os

# Measure Solution V2 deterministic consistency
COMPARATIVE_CONSISTENCY_RUNS = 3

v2_consistency_results = [
    run_solution_v2(
        query=test_query,
        k=V2_TOP_K
    )
    for _ in range(
        COMPARATIVE_CONSISTENCY_RUNS
    )
]

v2_consistency_outputs = [
    result["final_answer"]
    for result in v2_consistency_results
]

v2_consistency_router_outputs = [
    result["router_raw_output"]
    for result in v2_consistency_results
]

v2_consistency_sources = [
    result["retrieved_source"]
    for result in v2_consistency_results
]

v2_consistency_rate = (
    v2_consistency_outputs.count(
        v2_consistency_outputs[0]
    )
    / COMPARATIVE_CONSISTENCY_RUNS
    * 100
)

v2_router_consistent = (
    len(
        set(
            v2_consistency_router_outputs
        )
    )
    == 1
)

v2_retrieval_consistent = (
    len(
        set(
            v2_consistency_sources
        )
    )
    == 1
)

# Aggregate helper functions
def boolean_percentage(column):
    return (
        comparative_results_full_df[
            column
        ]
        .astype(bool)
        .mean()
        * 100
    )


def mean_score(column):
    return float(
        comparative_results_full_df[
            column
        ].mean()
    )


baseline_consistency_rate = float(
    v1_metrics_final_df[
        "baseline_consistency_rate_percent"
    ].iloc[0]
)

v1_consistency_rate = float(
    v1_metrics_final_df[
        "naive_rag_consistency_rate_percent"
    ].iloc[0]
)

# Create the summary metric rows
SUMMARY_ROWS = [
    {
        "metric": "Execution Success",
        "evaluation_scope":
            "Common 50-row held-out generation sample",
        "evaluation_records": 50,
        "unit": "percent",
        "preferred_direction": "Higher",
        "baseline": boolean_percentage(
            "baseline_success"
        ),
        "solution_v1": boolean_percentage(
            "v1_success"
        ),
        "solution_v2": boolean_percentage(
            "v2_success"
        )
    },
    {
        "metric": "Final-Answer Format Adherence",
        "evaluation_scope":
            "Common 50-row held-out generation sample",
        "evaluation_records": 50,
        "unit": "percent",
        "preferred_direction": "Higher",
        "baseline": boolean_percentage(
            "baseline_format_valid"
        ),
        "solution_v1": boolean_percentage(
            "v1_format_valid"
        ),
        "solution_v2": boolean_percentage(
            "v2_format_valid"
        )
    },
    {
        "metric": "Functional Correctness",
        "evaluation_scope":
            "Common 50-row held-out generation sample",
        "evaluation_records": 50,
        "unit": "percent",
        "preferred_direction": "Higher",
        "baseline": boolean_percentage(
            "baseline_functionally_correct"
        ),
        "solution_v1": boolean_percentage(
            "v1_functionally_correct"
        ),
        "solution_v2": boolean_percentage(
            "v2_functionally_correct"
        )
    },
    {
        "metric": "Hallucination Frequency",
        "evaluation_scope":
            "Common 50-row held-out generation sample",
        "evaluation_records": 50,
        "unit": "percent",
        "preferred_direction": "Lower",
        "baseline": boolean_percentage(
            "baseline_hallucination_detected"
        ),
        "solution_v1": boolean_percentage(
            "v1_hallucination_detected"
        ),
        "solution_v2": boolean_percentage(
            "v2_hallucination_detected"
        )
    },
    {
        "metric": "Output Consistency",
        "evaluation_scope":
            "Three repeated deterministic runs",
        "evaluation_records": 3,
        "unit": "percent",
        "preferred_direction": "Higher",
        "baseline": baseline_consistency_rate,
        "solution_v1": v1_consistency_rate,
        "solution_v2": v2_consistency_rate
    },
    {
        "metric": "ROUGE-1",
        "evaluation_scope":
            "Common 50-row SOP-grounded references",
        "evaluation_records": 50,
        "unit": "score",
        "preferred_direction": "Higher",
        "baseline": mean_score(
            "baseline_rouge1"
        ),
        "solution_v1": mean_score(
            "v1_rouge1"
        ),
        "solution_v2": mean_score(
            "v2_rouge1"
        )
    },
    {
        "metric": "ROUGE-L",
        "evaluation_scope":
            "Common 50-row SOP-grounded references",
        "evaluation_records": 50,
        "unit": "score",
        "preferred_direction": "Higher",
        "baseline": mean_score(
            "baseline_rougeL"
        ),
        "solution_v1": mean_score(
            "v1_rougeL"
        ),
        "solution_v2": mean_score(
            "v2_rougeL"
        )
    },
    {
        "metric": "BLEU",
        "evaluation_scope":
            "Common 50-row SOP-grounded references",
        "evaluation_records": 50,
        "unit": "score",
        "preferred_direction": "Higher",
        "baseline": mean_score(
            "baseline_bleu"
        ),
        "solution_v1": mean_score(
            "v1_bleu"
        ),
        "solution_v2": mean_score(
            "v2_bleu"
        )
    },
    {
        "metric": "Top-1 SOP Source Agreement",
        "evaluation_scope":
            "Common 50-row SOP-grounded references",
        "evaluation_records": 50,
        "unit": "percent",
        "preferred_direction": "Higher",
        "baseline": np.nan,
        "solution_v1": boolean_percentage(
            "v1_source_match"
        ),
        "solution_v2": boolean_percentage(
            "v2_source_match"
        )
    },
    {
        "metric": "Strict Router JSON Format Adherence",
        "evaluation_scope":
            "Complete held-out router test split",
        "evaluation_records":
            len(v2_router_results_df),
        "unit": "percent",
        "preferred_direction": "Higher",
        "baseline": np.nan,
        "solution_v1": np.nan,
        "solution_v2": v2_format_adherence
    },
    {
        "metric": "Exact Intent Accuracy",
        "evaluation_scope":
            "Complete held-out router test split",
        "evaluation_records":
            len(v2_router_results_df),
        "unit": "percent",
        "preferred_direction": "Higher",
        "baseline": np.nan,
        "solution_v1": np.nan,
        "solution_v2":
            v2_exact_intent_accuracy
    },
    {
        "metric": "Fuzzy Intent Accuracy",
        "evaluation_scope":
            "Complete held-out router test split",
        "evaluation_records":
            len(v2_router_results_df),
        "unit": "percent",
        "preferred_direction": "Higher",
        "baseline": np.nan,
        "solution_v1": np.nan,
        "solution_v2":
            v2_fuzzy_intent_accuracy
    },
    {
        "metric": "Exact Category Accuracy",
        "evaluation_scope":
            "Complete held-out router test split",
        "evaluation_records":
            len(v2_router_results_df),
        "unit": "percent",
        "preferred_direction": "Higher",
        "baseline": np.nan,
        "solution_v1": np.nan,
        "solution_v2":
            v2_exact_category_accuracy
    },
    {
        "metric": "Adversarial Exact Intent Accuracy",
        "evaluation_scope":
            "Regex-derived adversarial subset",
        "evaluation_records":
            len(v2_adversarial_df),
        "unit": "percent",
        "preferred_direction": "Higher",
        "baseline": np.nan,
        "solution_v1": np.nan,
        "solution_v2":
            v2_adversarial_exact_accuracy
    },
    {
        "metric": "Adversarial Fuzzy Intent Accuracy",
        "evaluation_scope":
            "Regex-derived adversarial subset",
        "evaluation_records":
            len(v2_adversarial_df),
        "unit": "percent",
        "preferred_direction": "Higher",
        "baseline": np.nan,
        "solution_v1": np.nan,
        "solution_v2":
            v2_adversarial_fuzzy_accuracy
    }
]


comparative_results_summary_df = pd.DataFrame(
    SUMMARY_ROWS
)

# Calculate beneficial absolute and relative changes
def beneficial_change(
    earlier_value,
    later_value,
    preferred_direction
):
    if (
        pd.isna(earlier_value)
        or pd.isna(later_value)
    ):
        return np.nan

    if preferred_direction == "Lower":
        return earlier_value - later_value

    return later_value - earlier_value


def relative_beneficial_change(
    earlier_value,
    later_value,
    preferred_direction
):
    if (
        pd.isna(earlier_value)
        or pd.isna(later_value)
        or np.isclose(earlier_value, 0)
    ):
        return np.nan

    return (
        beneficial_change(
            earlier_value,
            later_value,
            preferred_direction
        )
        / abs(earlier_value)
        * 100
    )


comparative_results_summary_df[
    "baseline_to_v1_beneficial_change"
] = comparative_results_summary_df.apply(
    lambda row: beneficial_change(
        row["baseline"],
        row["solution_v1"],
        row["preferred_direction"]
    ),
    axis=1
)

comparative_results_summary_df[
    "v1_to_v2_beneficial_change"
] = comparative_results_summary_df.apply(
    lambda row: beneficial_change(
        row["solution_v1"],
        row["solution_v2"],
        row["preferred_direction"]
    ),
    axis=1
)

comparative_results_summary_df[
    "baseline_to_v2_beneficial_change"
] = comparative_results_summary_df.apply(
    lambda row: beneficial_change(
        row["baseline"],
        row["solution_v2"],
        row["preferred_direction"]
    ),
    axis=1
)

comparative_results_summary_df[
    "baseline_to_v1_relative_improvement_percent"
] = comparative_results_summary_df.apply(
    lambda row: relative_beneficial_change(
        row["baseline"],
        row["solution_v1"],
        row["preferred_direction"]
    ),
    axis=1
)

comparative_results_summary_df[
    "v1_to_v2_relative_improvement_percent"
] = comparative_results_summary_df.apply(
    lambda row: relative_beneficial_change(
        row["solution_v1"],
        row["solution_v2"],
        row["preferred_direction"]
    ),
    axis=1
)

comparative_results_summary_df[
    "baseline_to_v2_relative_improvement_percent"
] = comparative_results_summary_df.apply(
    lambda row: relative_beneficial_change(
        row["baseline"],
        row["solution_v2"],
        row["preferred_direction"]
    ),
    axis=1
)

The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:447: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info

In [32]:
# Define final output paths
COMPARATIVE_FULL_PATH = (
    NOTEBOOK_7_ARTIFACT_DIR
    / "Comparative_Results_Full.csv"
)

COMPARATIVE_SUMMARY_PATH = (
    NOTEBOOK_7_ARTIFACT_DIR
    / "Comparative_Results_Summary.csv"
)

NOTEBOOK_7_ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Save both CSVs atomically
TEMP_FULL_PATH = (
    COMPARATIVE_FULL_PATH.with_suffix(
        ".tmp"
    )
)

TEMP_SUMMARY_PATH = (
    COMPARATIVE_SUMMARY_PATH.with_suffix(
        ".tmp"
    )
)

comparative_results_full_df.to_csv(
    TEMP_FULL_PATH,
    index=False
)

comparative_results_summary_df.to_csv(
    TEMP_SUMMARY_PATH,
    index=False
)

os.replace(
    TEMP_FULL_PATH,
    COMPARATIVE_FULL_PATH
)

os.replace(
    TEMP_SUMMARY_PATH,
    COMPARATIVE_SUMMARY_PATH
)

# Reload and validate final deliverables
saved_comparative_full = pd.read_csv(
    COMPARATIVE_FULL_PATH
)

saved_comparative_summary = pd.read_csv(
    COMPARATIVE_SUMMARY_PATH
)

assert len(saved_comparative_full) == 50

assert len(saved_comparative_summary) == len(
    comparative_results_summary_df
)

assert (
    saved_comparative_full.columns.tolist()
    ==
    comparative_results_full_df.columns.tolist()
)

assert (
    saved_comparative_summary.columns.tolist()
    ==
    comparative_results_summary_df.columns.tolist()
)

# Display final confirmation
print("Final comparative deliverables saved")
print("=" * 72)
print(
    f"Per-row results:\n"
    f"{COMPARATIVE_FULL_PATH}"
)
print(
    f"\nAggregate summary:\n"
    f"{COMPARATIVE_SUMMARY_PATH}"
)

print("\nSaved table dimensions")
print("-" * 72)
print(
    f"Comparative_Results_Full.csv    : "
    f"{saved_comparative_full.shape}"
)
print(
    f"Comparative_Results_Summary.csv : "
    f"{saved_comparative_summary.shape}"
)

print("\nSolution V2 consistency validation")
print("-" * 72)
print(
    f"Final-answer consistency : "
    f"{v2_consistency_rate:.2f}%"
)
print(
    f"Router output consistent : "
    f"{v2_router_consistent}"
)
print(
    f"Retrieval consistent     : "
    f"{v2_retrieval_consistent}"
)

print("\nFinal comparative summary:")

display(
    saved_comparative_summary.round(4)
)

Final comparative deliverables saved
Per-row results:
/content/drive/MyDrive/Hybrid_RAG_Customer_Support/artifacts/notebook_7/Comparative_Results_Full.csv

Aggregate summary:
/content/drive/MyDrive/Hybrid_RAG_Customer_Support/artifacts/notebook_7/Comparative_Results_Summary.csv

Saved table dimensions
------------------------------------------------------------------------
Comparative_Results_Full.csv    : (50, 61)
Comparative_Results_Summary.csv : (15, 14)

Solution V2 consistency validation
------------------------------------------------------------------------
Final-answer consistency : 100.00%
Router output consistent : True
Retrieval consistent     : True

Final comparative summary:


,metric,evaluation_scope,evaluation_records,unit,preferred_direction,baseline,solution_v1,solution_v2,baseline_to_v1_beneficial_change,v1_to_v2_beneficial_change,baseline_to_v2_beneficial_change,baseline_to_v1_relative_improvement_percent,v1_to_v2_relative_improvement_percent,baseline_to_v2_relative_improvement_percent
0,Execution Success,Common 50-row held-out generation sample,50,percent,Higher,100.0000,100.0000,100.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
1,Final-Answer Format Adherence,Common 50-row held-out generation sample,50,percent,Higher,100.0000,100.0000,100.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2,Functional Correctness,Common 50-row held-out generation sample,50,percent,Higher,54.0000,26.0000,38.0000,-28.0000,12.0000,-16.0000,-51.8519,46.1538,-29.6296
3,Hallucination Frequency,Common 50-row held-out generation sample,50,percent,Lower,46.0000,74.0000,62.0000,-28.0000,12.0000,-16.0000,-60.8696,16.2162,-34.7826
4,Output Consistency,Three repeated deterministic runs,3,percent,Higher,100.0000,100.0000,100.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
5,ROUGE-1,Common 50-row SOP-grounded references,50,score,Higher,0.2419,0.3058,0.3285,0.0640,0.0227,0.0867,26.4509,7.4171,35.8299
6,ROUGE-L,Common 50-row SOP-grounded references,50,score,Higher,0.1338,0.1708,0.1890,0.0370,0.0182,0.0552,27.6383,10.6655,41.2516
7,BLEU,Common 50-row SOP-grounded references,50,score,Higher,0.0123,0.0348,0.0338,0.0224,-0.0010,0.0214,181.7491,-2.8447,173.7343
8,Top-1 SOP Source Agreement,Common 50-row SOP-grounded references,50,percent,Higher,NaN,44.0000,84.0000,NaN,40.0000,NaN,NaN,90.9091,NaN
9,Strict Router JSON Format Adherence,Complete held-out router test split,400,percent,Higher,NaN,NaN,100.0000,NaN,NaN,NaN,NaN,NaN,NaN


---
## END-OF-NOTEBOOK CHECKLIST (FINAL)

> **IMPORTANT: This is the last graded notebook. Verify all deliverables.**

- [x] **4.3.1** LoRA merged + Hybrid RAG integration validated (intent → search → retrieve → generate)
- [x] **4.4.1** Format Adherence + Exact Match + Fuzzy Match on test split + adversarial subset
- [x] **4.4.2** Fine-tuning impact quantified (V1 vs V2 with %)
- [x] **4.5** All 3 architectures scored with SOP-grounded references
- [x] **`Comparative_Results_Full.csv` saved** ← _FINAL DELIVERABLE_
- [x] **`Comparative_Results_Summary.csv` saved** ← _FINAL DELIVERABLE_

### Complete Artifact Inventory

| Artifact | Created In |
|---|---|
| `sampled_data.csv` | NB1 |
| `./tokenized_train/`, `./tokenized_valid/`, `df_test.csv` | NB2 |
| `outputs.json` | NB3 + NB4 |
| `./chroma_db/` | NB4 |
| `v1_metrics.csv` | NB5 |
| `./intent_lora_best/`, `training_log.csv`, `training_curves.png` | NB6 |
| `Comparative_Results_Full.csv`, `Comparative_Results_Summary.csv` | NB7 |

**Mark all items checked, then prepare your final submission package.**